# GPU Metrics Data Processing & Imputation Notebook

This notebook performs end-to-end processing of GPU utilization metrics from Nessie/Iceberg tables using Spark on Enterprise Gateway (SparkCaster). It includes feature engineering, ML-based imputation of missing `redfish_power` values, and writes the results back to Nessie.

---
#sparkcaster using "http://spark-gateway.kubeflow.svc.cluster.local:8888"

## 📋 Notebook Structure

### **Setup & Initialization** (Cells 1-5)

- **Cell 1**: Package installation - Installs required Python packages including PySpark, Nessie/Iceberg connectors, visualization libraries, and ML tools
- **Cell 2**: Credential management - Sets up encrypted keyring for secure storage of Nessie credentials
- **Cell 3**: Credential validation - Verifies Nessie connection and authentication
- **Cell 4**: Spark cluster setup - Configures SparkCaster session with Nessie/Iceberg catalog, cleans up existing sessions, and initializes Spark context
- **Cell 5**: Data loading - Queries source data from Nessie using Spark SQL

### **Feature Engineering** (Cells 6-7)

- **Cell 6**: Feature engineering pipeline - Creates derived features for modeling and analysis:
  - `tensor_to_gpu_ratio` - Tensor utilization relative to GPU utilization
  - `memory_intensity` - Memory copy utilization relative to GPU utilization
  - `compute_to_memory_ratio` - Compute throughput relative to memory usage
  - `power_%_max` - Power usage as percentage of maximum
  - `compute_%_max` - Compute usage as percentage of maximum
  - `gen` - GPU generation classification (current_gen vs prior_gen)
  - `training_inference` - Simple workload classification:
    - **active_training**: `is_training == "slurm"`
    - **non_training**: `is_training != "slurm"`
    
- **Cell 7**: DataFrame creation - Prepares final DataFrame with all features for modeling

### **ML Model Training & Evaluation** (Cells 8-12)

- **Cell 8**: Complete ML pipeline with 5-fold cross-validation - Trains and evaluates multiple regression models to predict `redfish_power`:
  - Model 1: Simple linear regression (`chip_power` only)
  - Model 2: Multiple features (`chip_power`, `gpu_util`, `tensor_util`, `tensor_tflops`)
  - Uses cross-validation to select best model
  - Caches training data for performance
  
- **Cell 9**: Model 1 training - Simple linear regression baseline using only `chip_power`
- **Cell 10**: Diagnostic plots - Generates QQ plots and predicted vs actual visualizations for all trained models
- **Cell 11**: Memory cleanup - Unpersists cached data to free memory
- **Cell 12**: Final model training - Retrains the best performing model on optimized sample size (10% of data) for production use

### **Data Imputation & Output** (Cells 13-15)

- **Cell 13**: Imputation execution - Applies the trained model to predict missing `redfish_power` values across the full dataset
  - Identifies rows with missing values
  - Applies ML model predictions
  - Creates `redfish_power_imputed` flag for tracking
  - Combines imputed and original data
  - Reports imputation statistics
  
- **Cell 14**: Schema verification - Prints final DataFrame schema after imputation

- **Cell 15**: Write to Nessie - Writes the complete imputed dataset to Nessie/Iceberg table
  - **Target table**: `sandbox.sandbox_finance.dcgm_metrics_raw_impute`
  - Includes all original columns plus engineered features
  - Uses Iceberg table format for time-travel and ACID properties
  - Reports write statistics (row count, duration, throughput)

### **Analysis & Visualization** (Cells 16-17)

- **Cell 16**: Classification analysis - Analyzes the `training_inference` feature distribution and relationships with power metrics
- **Cell 17**: Classification visualization - Creates plots showing training vs inference workload patterns

### **Data Validation & Testing** (Cells 18+)

- **Cells 18+**: Data validation and testing - Verification queries to confirm successful write and data quality

---

## 🎯 Key Features

- **Feature Engineering**: Creates 7 derived features including GPU generation, workload classification, and performance ratios
- **Simplified Classification**: Binary workload classification (active_training vs non_training) based on Slurm job detection
- **ML-based Imputation**: Uses Spark MLlib with cross-validation to select best regression model for missing power values
- **Scalable Processing**: Designed for large datasets using Spark 3.5.5 on Enterprise Gateway (SparkCaster)
- **Nessie/Iceberg Output**: Writes to Iceberg tables for versioning, time-travel, and ACID guarantees
- **Secure Credentials**: Encrypted keyring storage for database passwords
- **Data Validation**: Built-in schema verification and row count checks

---

## 🚀 Quick Start

1. **Setup** (Cells 1-4): Install packages, configure credentials, initialize SparkCaster
2. **Load Data** (Cell 5): Query source data from Nessie
3. **Feature Engineering** (Cells 6-7): Create derived features including simplified training/inference classification
4. **Train Models** (Cells 8-12): Train ML models with cross-validation, select best model
5. **Impute** (Cells 13-14): Apply model to predict missing `redfish_power` values
6. **Write Output** (Cell 15): Save imputed dataset to Nessie/Iceberg table
7. **Analyze** (Cells 16-17): Explore workload classification patterns
8. **Validate** (Cells 18+): Verify data quality and row counts

---

## 📊 Key Data Assets

### Input Data
- **Source**: Nessie/Iceberg table via Spark SQL
- **Catalog**: `sandbox.sandbox_finance`
- **Processing**: In-memory Spark DataFrames with distributed processing

### Output Data
- **Target Table**: `sandbox.sandbox_finance.dcgm_metrics_raw_impute`
- **Format**: Apache Iceberg (enables time-travel, schema evolution, ACID transactions)
- **Features**: 21+ columns including:
  - Original metrics: `datestamp`, `node`, `gpu_util`, `tensor_util`, `chip_power`, `redfish_power`
  - Engineered features: `tensor_to_gpu_ratio`, `memory_intensity`, `compute_to_memory_ratio`, `power_%_max`, `compute_%_max`, `gen`, `training_inference`
  - Imputation metadata: `redfish_power_imputed` flag

### ML Features for Imputation
Primary model predictors (selected via cross-validation):
- `chip_power` - GPU chip power consumption (primary predictor)
- `gpu_util` - GPU utilization percentage
- `tensor_util` - Tensor core utilization
- `tensor_tflops` - Tensor FLOPS performance

### Workload Classification Logic
Simple binary classification for `training_inference` feature:
- **active_training**: Records where `is_training == "slurm"` (GPU actively running Slurm training jobs)
- **non_training**: All other records (`is_training != "slurm"`)

---

## 🔧 Technical Configuration

### SparkCaster / Enterprise Gateway
- **Endpoint**: `http://spark-gateway.kubeflow.svc.cluster.local:8888`
- **Spark Version**: 3.5.5
- **Catalog**: Nessie with Iceberg tables
- **Processing**: Distributed Spark DataFrames
- **Caching**: Strategic caching for model training performance

### ML Configuration
- **Framework**: Spark MLlib
- **Validation**: 5-fold cross-validation
- **Models Tested**: Simple and multi-feature linear regression
- **Training Sample**: 10% of data (optimized for performance vs accuracy)
- **Evaluation Metrics**: RMSE, R², MAE

### Data Output
- **Format**: Apache Iceberg (via Nessie catalog)
- **Write Mode**: Append/Overwrite (configurable)
- **Validation**: Row count verification after write
- **Benefits**: Time-travel, schema evolution, ACID properties, efficient updates

---

## ⚠️ Important Notes

- Feature engineering creates the simplified `training_inference` binary classification (active_training / non_training)
- The notebook uses cross-validation to automatically select the best performing ML model
- All imputed `redfish_power` values are flagged with `redfish_power_imputed` for traceability
- Nessie/Iceberg provides versioning - you can query previous versions of the table using time-travel
- Strategic data caching is used during model training to improve performance
- The complete pipeline is designed to run end-to-end without manual intervention

In [1]:
#install needed packages

import importlib
import subprocess
import sys

def is_module_available(module_name: str) -> bool:
    """Return True if the importable module exists; handles dotted names safely."""
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

def install_if_missing(pip_name: str, import_name: str = None):
    """
    Install `pip_name` only if `import_name` (or `pip_name` if None) is not importable.
    Use this to handle cases where pip/distribution names differ from import names.
    """
    import_name = import_name or pip_name
    if not is_module_available(import_name):
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
    else:
        print(f"{pip_name} already installed.")

# --- Packages for Nessie/Iceberg access ---
# simple 1:1 cases
install_if_missing("keyring", "keyring")
install_if_missing("ipython-secrets", "ipython_secrets")
install_if_missing("pyarrow", "pyarrow")
install_if_missing("fsspec", "fsspec")
install_if_missing("s3fs", "s3fs")
install_if_missing("statsmodels", "statsmodels")
install_if_missing("matplotlib", "matplotlib")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("xgboost", "xgboost")

# special cases
# keyrings.cryptfile installs a plugin under the 'keyrings' package; checking the root avoids ModuleNotFoundError
install_if_missing("keyrings.cryptfile", "keyrings")

# Core viz + scaling pins for Python 3.11.11
install_if_missing("bokeh==3.6.2", "bokeh")
install_if_missing("jupyter_bokeh", "jupyter_bokeh")
install_if_missing("panel==1.5.2", "panel")
install_if_missing("holoviews==1.19.0", "holoviews")
install_if_missing("hvplot==0.10.0", "hvplot")
install_if_missing("datashader==0.16.3", "datashader")
install_if_missing("dask[dataframe]==2024.9.1", "dask")
install_if_missing("distributed==2024.9.1", "distributed")
install_if_missing("reportlab", "reportlab")


print("\n✓ All packages installed/verified")




keyring already installed.
ipython-secrets already installed.
pyarrow already installed.
fsspec already installed.
s3fs already installed.
Installing statsmodels ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 39.9 MB/s  0:00:00ta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]
Installing matplotlib ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 40.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 250.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 179.9 MB/s  0:00:00
   ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [fonttools]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]
scikit-learn already installed.
Installing xgboost ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 146.0 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 170.2 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]
keyrings.cryptfile already installed.
Installing bokeh==3.6.2 ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 9.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [bokeh]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bokeh]
Installing jupyter_bokeh ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.4 MB/s  0:00:00 eta 0:00:01
Installing panel==1.5.2 ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 52.4 MB/s  0:00:00m0:00:010:02
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 3/6 [bleach]

...

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of dask-expr to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.9 MB/s  0:00:00m eta 0:00:01
  Attempting uninstall: dask
    Found existing installation: dask 2026.7.1
    Uninstalling dask-2026.7.1:
      Successfully uninstalled dask-2026.7.1
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [dask-expr]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dask-expr]
Installing distributed==2024.9.1 ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [distributed] [distributed]
Installing reportlab ...


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.4 MB/s  0:00:00eta 0:00:01

✓ All packages installed/verified


In [2]:
#set up caios credentials
import keyring
import os
from getpass import getpass
from keyrings.cryptfile.cryptfile import CryptFileKeyring
from pathlib import Path

# Use home directory dynamically instead of hardcoded path
home_dir = str(Path.home())
os.environ["KEYRING_CRYPTFILE_PATH"] = f"{home_dir}/.local/share/python_keyring/cryptfile_pass.cfg"

kr = CryptFileKeyring()
kr.keyring_key = getpass("Set/enter master password for encrypted keyring: ")
keyring.set_keyring(kr)

# Prompt for CAIOS credentials if not already stored
caios_access_key = keyring.get_password("caios", "access_key")
caios_secret_key = keyring.get_password("caios", "secret_key")  

if not caios_access_key:
    caios_access_key = input("Enter CAIOS access key: ")
    keyring.set_password("caios", "access_key", caios_access_key)

if not caios_secret_key:
    caios_secret_key = getpass("Enter CAIOS secret key: ")
    keyring.set_password("caios", "secret_key", caios_secret_key)

print("✓ CAIOS credentials configured")

# ========================================
# StarRocks Credentials Setup
# ========================================
print("="*60)
print("StarRocks Credentials Setup")
print("="*60)

# Prompt for StarRocks credentials if not already stored
starrocks_username = keyring.get_password("starrocks", "username")
starrocks_password = keyring.get_password("starrocks", "password")

if not starrocks_username:
    starrocks_username = input("Enter StarRocks username: ")
    keyring.set_password("starrocks", "username", starrocks_username)

if not starrocks_password:
    starrocks_password = getpass("Enter StarRocks password: ")
    keyring.set_password("starrocks", "password", starrocks_password)

print("✓ StarRocks credentials configured")


✓ CAIOS credentials configured
StarRocks Credentials Setup
✓ StarRocks credentials configured


In [3]:
#credential validation
import keyring

# Retrieve CAIOS credentials from keyring
caios_access_key = keyring.get_password("caios", "access_key")
caios_secret_key = keyring.get_password("caios", "secret_key")

assert caios_access_key and caios_secret_key, "No CAIOS credentials in keyring."

print("✓ CAIOS credentials validated")
print(f"  Access key: {caios_access_key[:3]}...{caios_access_key[-3:]}")

✓ CAIOS credentials validated
  Access key: CWO...SKY


In [4]:
# CLEAN UP EXISTING SESSION
# ========================================
from spark.nessie.client import NessieSparkClient


try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        print("Stopping existing Spark session...")
        existing_spark.stop()
        print("✓ Existing Spark session stopped")
    else:
        print("No existing Spark session found")
except Exception as e:
    print(f"Note: Could not check for existing session: {e}")

ness = NessieSparkClient(
    svc_url="http://kf-proxy.nessie.svc.cluster.local:19120/api/v2",
    nessie_endpoint="http://nessie-prd.cwobject.com",
    caios_access_key=caios_access_key,
    caios_secret_key=caios_secret_key,
    dbtcaster=True,
)

spark = ness.spark

spark.sparkContext.setLogLevel("ERROR")

# ========================================
# SESSION DIAGNOSTICS
# ========================================
print("=" * 60)
print("SPARK CLUSTER CONFIGURATION - NESSIE SPARK CLIENT")
print("=" * 60)
print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")
print(f"App Name: {spark.sparkContext.appName}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print("Nessie Service URL: http://kf-proxy.nessie.svc.cluster.local:19120/api/v2")
print("Nessie Endpoint: http://nessie-prd.cwobject.com")
print("DBTCaster Mode: Enabled")
print("AWS Environment Variables: Set (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)")

for k in [
    "spark.driver.memory",
    "spark.driver.cores",
    "spark.executor.instances",
    "spark.executor.cores",
    "spark.executor.memory",
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
]:
    try:
        print(f"{k}: {spark.conf.get(k)}")
    except Exception:
        print(f"{k}: <unset>")

try:
    executor_count = spark.sparkContext._jsc.sc().getExecutorMemoryStatus().size()
    print(f"Executor entries visible: {executor_count}")
except Exception as e:
    print(f"Could not inspect executor status: {e}")

print("=" * 60)



Note: Could not check for existing session: name 'SparkSession' is not defined
SPARK CLUSTER CONFIGURATION - NESSIE SPARK CLIENT
Spark Version: 4.1.2
Spark Master: k8s://https://10.16.0.1:443
App Name: SparkClient for Nessie Catalog (and Iceberg)
Application ID: spark-0b1fa80acdc24630b07f16de5dffbf3b
Nessie Service URL: http://kf-proxy.nessie.svc.cluster.local:19120/api/v2
Nessie Endpoint: http://nessie-prd.cwobject.com
DBTCaster Mode: Enabled
AWS Environment Variables: Set (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)
spark.driver.memory: 16g
spark.driver.cores: 4
spark.executor.instances: 10
spark.executor.cores: 4
spark.executor.memory: 64g
spark.sql.shuffle.partitions: 200
spark.sql.adaptive.enabled: true
Executor entries visible: 5


In [5]:
#query data
df = ness.sql("select * from nessie.staging_capacity_finance.dcgm_metrics_raw")
#df = ness.sql("select max(datestamp) from nessie.staging_capacity_finance.dcgm_metrics_raw")
df.limit(5).show(truncate=False)
df.printSchema()


+-------------------+-------+--------+-----------+-----------------+-------------------+-------------+----------+----------+--------+------------+------+-----+-------+-----------+------+---------------------+------+-------------+-------------+-----------------+---------------------+-------+----------------+----------------+-----------+------------+------------------+----------------+---------------+---------------+----------------------+--------------------------+-----------------------+----------------------+-----------------+-------------+-------------------------+
|datestamp          |node   |gpu_util|tensor_util|chip_power       |dram_active        |mem_copy_util|vram_usage|sm_active |sm_clock|sm_occupancy|region|zone |cluster|cluster_org|cw_sku|model                |serial|redfish_power|customer_name|flag_is_coreweave|model_imputed        |product|product_resolved|customer_segment|is_training|is_multinode|gpu_count_expected|peak_tflops_unit|peak_power_unit|product_segment|memory_b

In [6]:
# FEATURE ENGINEERING
# ========================================
# Add derived features to the raw dataset
# These features will be available for EDA, modeling, and imputation

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("="*60)
print("FEATURE ENGINEERING")
print("="*60)

print(f"\nStarting with {len(df.columns)} columns")

# Get initial count
# initial_count = df.count()
# print(f"Row count: {initial_count:,}")

# ========================================
# 1. Tensor to GPU Ratio
# ========================================
print("\n[1/5] Adding tensor_to_gpu_ratio...")
df = df.withColumn(
    "tensor_to_gpu_ratio_index",
    F.col("tensor_util") / (F.col("gpu_util") + 0.0001)
)
print("✓ tensor_to_gpu_ratio = tensor_util / (gpu_util + 0.001)")

# ========================================
# 2. Memory Intensity
# ========================================
print("\n[2/5] Adding memory_intensity...")
df = df.withColumn(
    "memory_intensity_index",
    F.col("mem_copy_util") / (F.col("gpu_util") + 0.0001)
)
print("✓ memory_intensity = mem_copy_util / (gpu_util + 0.001)")

# ========================================
# 3. Compute to Memory Ratio
# ========================================
print("\n[3/5] Adding compute_to_memory_ratio...")
df = df.withColumn(
    "compute_to_memory_ratio_index",
    F.col("tensor_tflops") / (F.col("vram_usage") + 0.0001)
)
print("✓ compute_to_memory_ratio = tensor_tflops / (vram_usage + 0.001)")

# ========================================
# 4. Power % of Max
# ========================================
print("\n[4/7] Adding power_%_max...")
df = df.withColumn(
    "power_pct_max_index",
    F.col("chip_power") / (F.col("gpu_peak_power_node_watts") + 0.0001)
)
print("✓ power_%_max = chip_power / (gpu_peak_power_node_watts + 0.001)")

# ========================================
# 5. Compute % of Max
# ========================================
print("\n[5/7] Adding compute_%_max...")
df = df.withColumn(
    "compute_pct_max_index",
    F.col("tensor_tflops") / ((F.col("peak_tflops_unit") * F.col("gpu_count_expected")) + 0.0001)
)
print("✓ compute_%_max = tensor_tflops / ((peak_tflops_unit * gpu_count_expected) + 0.001)")

# ========================================
# 6. GPU Generation
# ========================================
print("\n[6/7] Adding gen (GPU generation)...")
df = df.withColumn(
    "gen",
    F.when(
        F.col("product_resolved").isin("GB200", "GB300", "B200", "B300"),
        "current_gen"
    ).otherwise("prior_gen")
)
print("✓ gen = 'current_gen' if product_resolved in ('GB200', 'GB300', 'B200', 'B300'), else 'prior_gen'")

# Show distribution
print("\nGPU generation distribution:")
df.groupBy("gen").count().orderBy("gen").show()


# ========================================
# 7. Region Rollup
# ========================================
print("\n[7/8] Adding region_rollup...")

# Normalize region: trim whitespace, handle nulls, create region_rollup
df = df.withColumn(
    "region_rollup",
    F.when(
        F.trim(F.coalesce(F.col("region"), F.lit(""))).startswith("EU"),
        "EU"
    ).otherwise("NAM")
)

print("✓ region_rollup = 'EU' if region starts with 'EU', else 'NAM'")

# Show distribution
print("\nRegion rollup distribution:")
df.select("region", "region_rollup").distinct().orderBy("region").show(50, truncate=False)

# ========================================
# 8. Training/Inference Classification
# ========================================
print("\n[7/7] Adding training_inference classification...")

# Simple classification: if is_training == "slurm" then "active_training", else "non_training"
df = df.withColumn(
    "training_inference",
    F.when(
        F.col("is_training") == "slurm",
        "active_training"
    ).otherwise(
        "non_training"
    )
)

print("✓ training_inference classification complete:")
print("  - 'active_training': is_training = 'slurm'")
print("  - 'non_training': is_training != 'slurm'")

# Show distribution
# print("\nTraining/Inference distribution:")
# df.groupBy("training_inference").count().orderBy("count", ascending=False).show()

# ========================================
# Summary
# ========================================
print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE")
print("="*60)

# print(f"\nFinal column count: {len(df.columns)} columns")
print(f"Added features:")
print("  1. tensor_to_gpu_ratio")
print("  2. memory_intensity")
print("  3. compute_to_memory_ratio")
print("  4. power_%_max")
print("  5. compute_%_max")
print("  6. gen")
print("  7. region_rollup")
print("  8. training_inference")

# Show sample with new features
print("\nSample with new features:")
df.select(
    "datestamp", "node", "region", "region_rollup", "product_resolved", "gen",
    "tensor_to_gpu_ratio_index", "memory_intensity_index", "compute_to_memory_ratio_index",
    "power_pct_max_index", "compute_pct_max_index",
    "is_training", "training_inference", "redfish_power"
).show(10, truncate=False)

print("\n✓ Features are now available for all downstream analysis and modeling!")

FEATURE ENGINEERING

Starting with 38 columns

[1/5] Adding tensor_to_gpu_ratio...
✓ tensor_to_gpu_ratio = tensor_util / (gpu_util + 0.001)

[2/5] Adding memory_intensity...
✓ memory_intensity = mem_copy_util / (gpu_util + 0.001)

[3/5] Adding compute_to_memory_ratio...
✓ compute_to_memory_ratio = tensor_tflops / (vram_usage + 0.001)

[4/7] Adding power_%_max...
✓ power_%_max = chip_power / (gpu_peak_power_node_watts + 0.001)

[5/7] Adding compute_%_max...
✓ compute_%_max = tensor_tflops / ((peak_tflops_unit * gpu_count_expected) + 0.001)

[6/7] Adding gen (GPU generation)...
✓ gen = 'current_gen' if product_resolved in ('GB200', 'GB300', 'B200', 'B300'), else 'prior_gen'

GPU generation distribution:
+-----------+---------+
|        gen|    count|
+-----------+---------+
|current_gen|414080397|
|  prior_gen|704409913|
+-----------+---------+


[7/8] Adding region_rollup...
✓ region_rollup = 'EU' if region starts with 'EU', else 'NAM'

Region rollup distribution:
+-------------+-------

In [7]:
#dataframe creation
from pyspark.sql import functions as F, types as T
from pyspark.sql.functions import monotonically_increasing_id

# 0) Pick only the numeric columns you care about (cast to double for safety)
num_cols = ["tensor_util","gpu_util","tensor_tflops","chip_power","redfish_power",
            "peak_power_unit","gpu_peak_power_node_watts","peak_tflops_unit",
            "tensor_tflops_sm" , "sm_active", "sm_clock", "sm_occupancy", "dram_active",
            "mem_copy_util","vram_usage","TFLOPS_per_watt_efficiency",
            # "compute_occupancy_index","tensor_util_dram_index","memory_boundness_index", #comment out as indexes are calculations of base features and not needed for null power impute
            "tensor_to_gpu_ratio_index","memory_intensity_index","power_pct_max_index", "compute_to_memory_ratio_index", "compute_pct_max_index" # new features added from feature engineering above
            ]


sdf_num = (
    df
      .select(*num_cols)
      .na.drop() #(subset=["redfish_power"])  # Only drop rows where target is NULL
      .select(*(F.col(c).cast(T.DoubleType()).alias(c) for c in num_cols))
)

# sdf_num_refactor = (
#     df.select(*num_cols, "_row_id")  # Include existing _row_id
#       .select(*(F.col(c).cast(T.DoubleType()).alias(c) for c in num_cols), "_row_id")
# )

# print(f"✓ Created sdf_num with {num_cols}")
# print("✓ Created sdf_num_refactor with _row_id column")

# ========================================
# Cache sdf_num for reuse across cells
# ========================================
# sdf_num is used by Cell 8 (initial training) and Cell 12 (final training)
# Caching avoids re-reading 900M rows from Nessie each time
print("\n" + "="*60)
print("CACHING sdf_num FOR TRAINING CELLS")
print("="*60)

sdf_num.cache()
sdf_num_count = sdf_num.count()  # Materialize the cache

# Estimate memory usage (columns already Double in Nessie)
rows = sdf_num_count
cols = len(num_cols)
estimated_gb = (rows * cols * 8) / (1024**3)

print(f"✓ Cached sdf_num: {sdf_num_count:,} rows × {cols} columns")
print(f"  Estimated memory: ~{estimated_gb:.1f} GB (raw), ~{estimated_gb*2:.1f} GB (with overhead)")
print(f"  Available executor memory: 640 GB")
print(f"  Cache usage: ~{(estimated_gb*2/640)*100:.1f}% of executor memory")
print("="*60)



CACHING sdf_num FOR TRAINING CELLS
✓ Cached sdf_num: 777,181,448 rows × 21 columns
  Estimated memory: ~121.6 GB (raw), ~243.2 GB (with overhead)
  Available executor memory: 640 GB
  Cache usage: ~38.0% of executor memory


In [8]:
# SPARK MLLIB MODEL TRAINING PIPELINE (OPTIMIZED + 5-FOLD CV)
# ========================================
# Performance optimizations:
# 1) Persist train/test datasets once (no repeated recomputation)
# 2) Persist assembled features once for tree models
# 3) Avoid unnecessary .count() calls
# 4) Smart sampling: 2% for quick iteration, 10% for final training
# 5) Tuned hyperparameters for speed on large data
# 6) 5-fold cross-validation for robust performance estimates

import time
from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler as MLStandardScaler
from pyspark.ml.regression import LinearRegression as MLLinearRegression, GBTRegressor, RandomForestRegressor as MLRandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import functions as F


print("="*60)
print("SPARK MLLIB TRAINING PIPELINE (OPTIMIZED + 5-FOLD CV)")
print("="*60)

# ========================================
# CONFIGURATION
# ========================================
SAMPLE_FRAC = 0.02  # Start with 3% for quick model selection
SEED = 42
NUM_FOLDS = 5       # 5-fold cross-validation

feature_cols = [c for c in num_cols if c != "redfish_power"]
target_col = "redfish_power"

print(f"\nFeature columns: {feature_cols}")
print(f"Target column: {target_col}")
print(f"Cross-validation: {NUM_FOLDS}-fold")

# ========================================
# Data Preparation with Persistence - FIXED for deterministic splits
# ========================================
print("\n[1/4] Preparing data with persistence...")

# Keep only necessary columns early (filter out nulls in target)
base_labeled = sdf_num.select(*(feature_cols + [target_col])) \
    .where(F.col(target_col).isNotNull())

# CRITICAL FIX: Add row ID before splitting to ensure deterministic train/test
# This prevents non-deterministic sampling issues where test R² > train R²
base_labeled = base_labeled.withColumn("_split_id", F.monotonically_increasing_id())

# SPLIT FIRST, then sample (this is the correct order!)
# Ensures train/test are truly from the same population distribution
train_sdf, test_sdf = base_labeled.randomSplit([0.7, 0.3], seed=SEED)

# Sample AFTER split (preserves train/test distribution)
if SAMPLE_FRAC < 1.0:
    train_sdf = train_sdf.sample(False, SAMPLE_FRAC, SEED)
    test_sdf = test_sdf.sample(False, SAMPLE_FRAC, SEED + 1)  # Different seed for test
    print(f"Split data first, then sampled {SAMPLE_FRAC*100:.0f}% from each")
else:
    print("Using full labeled dataset")

# Drop the split ID
train_sdf = train_sdf.drop("_split_id")
test_sdf = test_sdf.drop("_split_id")

# CRITICAL: Persist and materialize once
train_sdf.cache()
test_sdf.cache()

# Materialize once (single count job, not per model)
train_count = train_sdf.count()
test_count = test_sdf.count()

print(f"Training set: {train_count:,} rows (cached)")
print(f"Test set: {test_count:,} rows (cached)")
print(f"Num partitions: train={train_sdf.rdd.getNumPartitions()}, test={test_sdf.rdd.getNumPartitions()}")

# ========================================
# Feature Assembly with Persistence
# ========================================
print("\n[2/4] Pre-assembling features...")

# Assemble features once for ALL models
assembler_all = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)

# Pre-compute and cache assembled features for tree models
assembled_train = assembler_all.transform(train_sdf) \
    .select("features_raw", target_col)
assembled_train.cache()

assembled_test = assembler_all.transform(test_sdf) \
    .select("features_raw", target_col)
assembled_test.cache()

# Materialize assembled features
_ = assembled_train.count()
_ = assembled_test.count()

print("✓ Features assembled and cached")

# StandardScaler for linear models (withMean=False to avoid densification)
scaler = MLStandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withStd=True,
    withMean=False  # ← Avoids densifying sparse vectors, saves memory
)

print("✓ StandardScaler configured (withMean=False for memory efficiency)")

# # Fit scaler and create scaled features for GBT/RF
# scaler_model = scaler.fit(assembled_train)
# assembled_train_scaled = scaler_model.transform(assembled_train) \
#     .select("features_raw", "features_scaled", target_col)
# assembled_train_scaled.cache()

# assembled_test_scaled = scaler_model.transform(assembled_test) \
#     .select("features_raw", "features_scaled", target_col)
# assembled_test_scaled.cache()

# # Materialize scaled features
# _ = assembled_train_scaled.count()
# _ = assembled_test_scaled.count()

# print("✓ Scaled features computed and cached")

# ========================================
# Evaluation Setup
# ========================================
evaluator_r2 = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="mae")

# Double-check caching worked
print(f"\n✓ Caching status:")
print(f"  train_sdf: {train_sdf.is_cached}")
print(f"  test_sdf: {test_sdf.is_cached}")
print(f"  assembled_train: {assembled_train.is_cached}")
print(f"  assembled_test: {assembled_test.is_cached}")
# print(f"  assembled_train_scaled: {assembled_train_scaled.is_cached}")
# print(f"  assembled_test_scaled: {assembled_test_scaled.is_cached}")





SPARK MLLIB TRAINING PIPELINE (OPTIMIZED + 5-FOLD CV)

Feature columns: ['tensor_util', 'gpu_util', 'tensor_tflops', 'chip_power', 'peak_power_unit', 'gpu_peak_power_node_watts', 'peak_tflops_unit', 'tensor_tflops_sm', 'sm_active', 'sm_clock', 'sm_occupancy', 'dram_active', 'mem_copy_util', 'vram_usage', 'TFLOPS_per_watt_efficiency', 'tensor_to_gpu_ratio_index', 'memory_intensity_index', 'power_pct_max_index', 'compute_to_memory_ratio_index', 'compute_pct_max_index']
Target column: redfish_power
Cross-validation: 5-fold

[1/4] Preparing data with persistence...
Split data first, then sampled 2% from each
Training set: 10,878,946 rows (cached)
Test set: 4,662,389 rows (cached)
Num partitions: train=692, test=692

[2/4] Pre-assembling features...
✓ Features assembled and cached
✓ StandardScaler configured (withMean=False for memory efficiency)

✓ Caching status:
  train_sdf: True
  test_sdf: True
  assembled_train: True
  assembled_test: True


In [9]:
# Model 1: Simple Linear Regression (chip_power only)
# ========================================
print("\n[1/4] Training Simple Linear Regression (chip_power only)...")

assembler_chip = VectorAssembler(
    inputCols=["chip_power"],
    outputCol="features_chip"
)

simple_lr = MLLinearRegression(
    featuresCol="features_chip",
    labelCol=target_col,
    predictionCol="prediction",
    maxIter=100,
    standardization=False
)

simple_lr_pipeline = Pipeline(stages=[assembler_chip, simple_lr])

simple_lr_start = time.time()
simple_lr_model = simple_lr_pipeline.fit(train_sdf)
simple_lr_time = time.time() - simple_lr_start

# Evaluate on train/test
simple_lr_train_pred = simple_lr_model.transform(train_sdf)
simple_lr_test_pred = simple_lr_model.transform(test_sdf)

simple_lr_train_r2 = evaluator_r2.evaluate(simple_lr_train_pred)
simple_lr_train_rmse = evaluator_rmse.evaluate(simple_lr_train_pred)
simple_lr_train_mae = evaluator_mae.evaluate(simple_lr_train_pred)
simple_lr_test_r2 = evaluator_r2.evaluate(simple_lr_test_pred)
simple_lr_test_rmse = evaluator_rmse.evaluate(simple_lr_test_pred)
simple_lr_test_mae = evaluator_mae.evaluate(simple_lr_test_pred)

lr_stage = simple_lr_model.stages[-1]
print(f"  Training   → R²: {simple_lr_train_r2:.4f} | RMSE: {simple_lr_train_rmse:.2f} | MAE: {simple_lr_train_mae:.2f}")
print(f"  Test       → R²: {simple_lr_test_r2:.4f} | RMSE: {simple_lr_test_rmse:.2f} | MAE: {simple_lr_test_mae:.2f}")
print(f"  Coefficient: {lr_stage.coefficients[0]:.4f} | Intercept: {lr_stage.intercept:.2f}")
print(f"  Training time: {simple_lr_time:.2f}s")

# ========================================
# Model 2: Ridge Regression (all features, scaled)
# ========================================
print("\n[2/4] Training Ridge Regression (all features, scaled)...")

ridge_lr = MLLinearRegression(
    featuresCol="features_raw",
    labelCol=target_col,
    predictionCol="prediction",
    maxIter=100,
    regParam=0.01,  # Use default ridge regularization
    elasticNetParam=0.0,
    standardization=False
)

ridge_pipeline = Pipeline(stages=[scaler, ridge_lr])

ridge_start = time.time()
ridge_model = ridge_pipeline.fit(assembled_train)
ridge_time = time.time() - ridge_start

ridge_train_pred = ridge_model.transform(assembled_train)
ridge_test_pred = ridge_model.transform(assembled_test)

ridge_train_r2 = evaluator_r2.evaluate(ridge_train_pred)
ridge_train_rmse = evaluator_rmse.evaluate(ridge_train_pred)
ridge_train_mae = evaluator_mae.evaluate(ridge_train_pred)
ridge_test_r2 = evaluator_r2.evaluate(ridge_test_pred)
ridge_test_rmse = evaluator_rmse.evaluate(ridge_test_pred)
ridge_test_mae = evaluator_mae.evaluate(ridge_test_pred)

print(f"  Training   → R²: {ridge_train_r2:.4f} | RMSE: {ridge_train_rmse:.2f} | MAE: {ridge_train_mae:.2f}")
print(f"  Test       → R²: {ridge_test_r2:.4f} | RMSE: {ridge_test_rmse:.2f} | MAE: {ridge_test_mae:.2f}")
print(f"  Training time: {ridge_time:.2f}s")

# ========================================
# Model 3: Gradient Boosted Trees (with best CV params)
# ========================================
print("\n[3/4] Training Gradient Boosted Trees (with best CV params)...")

gbt = GBTRegressor(
    featuresCol="features_raw",
    labelCol=target_col,
    predictionCol="prediction",
    maxIter=30,
    maxDepth=12,             # ✓ Best param from CV
    stepSize=0.2,            # ✓ Best param from CV
    maxBins=32,
    subsamplingRate=0.6,
    minInstancesPerNode=100,
    seed=SEED
)

gbt_start = time.time()
gbt_model = gbt.fit(assembled_train)
gbt_time = time.time() - gbt_start

gbt_train_pred = gbt_model.transform(assembled_train)
gbt_test_pred = gbt_model.transform(assembled_test)

gbt_train_r2 = evaluator_r2.evaluate(gbt_train_pred)
gbt_train_rmse = evaluator_rmse.evaluate(gbt_train_pred)
gbt_train_mae = evaluator_mae.evaluate(gbt_train_pred)
gbt_test_r2 = evaluator_r2.evaluate(gbt_test_pred)
gbt_test_rmse = evaluator_rmse.evaluate(gbt_test_pred)
gbt_test_mae = evaluator_mae.evaluate(gbt_test_pred)

feature_importance = list(zip(feature_cols, gbt_model.featureImportances.toArray()))
feature_importance.sort(key=lambda x: x[1], reverse=True)

print(f"  Training   → R²: {gbt_train_r2:.4f} | RMSE: {gbt_train_rmse:.2f} | MAE: {gbt_train_mae:.2f}")
print(f"  Test       → R²: {gbt_test_r2:.4f} | RMSE: {gbt_test_rmse:.2f} | MAE: {gbt_test_mae:.2f}")
print(f"  Training time: {gbt_time:.2f}s")
print(f"  Hyperparameters: maxIter={gbt.getMaxIter()}, maxDepth={gbt.getMaxDepth()}, stepSize={gbt.getStepSize()}, subsample={gbt.getSubsamplingRate()}")
print(f"\n  Feature Importance:")
for feat, imp in feature_importance:
    print(f"    {feat}: {imp:.4f}")

# ========================================
# Model 4: Random Forest (with best CV params)
# ========================================
print("\n[4/4] Training Random Forest (with best CV params)...")

rf = MLRandomForestRegressor(
    featuresCol="features_raw",
    labelCol=target_col,
    predictionCol="prediction",
    numTrees=150,                     # ✓ Best param from CV
    maxDepth=15,                      # ✓ Best param from CV
    featureSubsetStrategy="sqrt",
    minInstancesPerNode=50,
    subsamplingRate=0.7,
    seed=SEED
)

rf_start = time.time()
rf_model = rf.fit(assembled_train)
rf_time = time.time() - rf_start

rf_train_pred = rf_model.transform(assembled_train)
rf_test_pred = rf_model.transform(assembled_test)

rf_train_r2 = evaluator_r2.evaluate(rf_train_pred)
rf_train_rmse = evaluator_rmse.evaluate(rf_train_pred)
rf_train_mae = evaluator_mae.evaluate(rf_train_pred)
rf_test_r2 = evaluator_r2.evaluate(rf_test_pred)
rf_test_rmse = evaluator_rmse.evaluate(rf_test_pred)
rf_test_mae = evaluator_mae.evaluate(rf_test_pred)

rf_feature_importance = list(zip(feature_cols, rf_model.featureImportances.toArray()))
rf_feature_importance.sort(key=lambda x: x[1], reverse=True)

print(f"  Training   → R²: {rf_train_r2:.4f} | RMSE: {rf_train_rmse:.2f} | MAE: {rf_train_mae:.2f}")
print(f"  Test       → R²: {rf_test_r2:.4f} | RMSE: {rf_test_rmse:.2f} | MAE: {rf_test_mae:.2f}")
print(f"  Training time: {rf_time:.2f}s")
print(f"  Hyperparameters: numTrees={rf.getNumTrees()}, maxDepth={rf.getMaxDepth()}, featureSubset={rf.getFeatureSubsetStrategy()}")
print(f"\n  Feature Importance:")
for feat, imp in rf_feature_importance:
    print(f"    {feat}: {imp:.4f}")

# ========================================
# Model Comparison
# ========================================
print("\n" + "="*60)
print("SPARK MLLIB MODEL COMPARISON")
print("="*60)

# Create results table
results = [
    ("Simple LR (chip_power)", simple_lr_train_r2, simple_lr_test_r2, simple_lr_test_rmse, simple_lr_test_mae, simple_lr_time),
    ("Ridge (all features)", ridge_train_r2, ridge_test_r2, ridge_test_rmse, ridge_test_mae, ridge_time),
    ("Gradient Boosting", gbt_train_r2, gbt_test_r2, gbt_test_rmse, gbt_test_mae, gbt_time),
    ("Random Forest", rf_train_r2, rf_test_r2, rf_test_rmse, rf_test_mae, rf_time)
]

print(f"{'Model':<25} {'Train R²':<10} {'Test R²':<10} {'Test RMSE':<12} {'Test MAE':<12} {'Overfit Gap':<12} {'Time (s)':<10}")
for model_name, train_r2, test_r2, test_rmse, test_mae, train_time in results:
    overfit_gap = train_r2 - test_r2
    print(f"{model_name:<25} {train_r2:<10.6f} {test_r2:<10.6f} {test_rmse:<12.2f} {test_mae:<12.2f} {overfit_gap:<12.6f} {train_time:<10.2f}")

# Find best model
best_model_idx = max(range(len(results)), key=lambda i: results[i][2])  # Test R²
best_model_name, _, best_test_r2, _, _, _ = results[best_model_idx]

print(f"\n✓ Best Model (based on Test R²): {best_model_name}")
print(f"  Test R²: {best_test_r2:.4f}")

total_time = sum(r[5] for r in results)
print(f"\n⚡ Total training time: {total_time:.2f}s")


# ========================================
# Save Models and Hyperparameters for Retraining
# ========================================
# Store models and their hyperparameters for later use
trained_models = {
    'Simple LR (chip_power)': {
        'model': simple_lr_model,
        'hyperparameters': {},  # No hyperparameters
        'metrics': {'train_r2': simple_lr_train_r2, 'test_r2': simple_lr_test_r2, 
                   'test_rmse': simple_lr_test_rmse, 'test_mae': simple_lr_test_mae}
    },
    'Ridge (all features)': {
        'model': ridge_model,
        'hyperparameters': {'regParam': ridge_lr.getRegParam(), 'elasticNetParam': ridge_lr.getElasticNetParam()},
        'metrics': {'train_r2': ridge_train_r2, 'test_r2': ridge_test_r2,
                   'test_rmse': ridge_test_rmse, 'test_mae': ridge_test_mae}
    },
    'Gradient Boosting': {
        'model': gbt_model,
        'hyperparameters': {
            'maxIter': gbt.getMaxIter(),
            'maxDepth': gbt.getMaxDepth(),
            'stepSize': gbt.getStepSize(),
            'subsamplingRate': gbt.getSubsamplingRate(),
            'minInstancesPerNode': gbt.getMinInstancesPerNode(),
            'maxBins': gbt.getMaxBins()
        },
        'metrics': {'train_r2': gbt_train_r2, 'test_r2': gbt_test_r2,
                   'test_rmse': gbt_test_rmse, 'test_mae': gbt_test_mae}
    },
    'Random Forest': {
        'model': rf_model,
        'hyperparameters': {
            'numTrees': rf.getNumTrees(),
            'maxDepth': rf.getMaxDepth(),
            'featureSubsetStrategy': rf.getFeatureSubsetStrategy(),
            'minInstancesPerNode': rf.getMinInstancesPerNode(),
            'subsamplingRate': rf.getSubsamplingRate()
        },
        'metrics': {'train_r2': rf_train_r2, 'test_r2': rf_test_r2,
                   'test_rmse': rf_test_rmse, 'test_mae': rf_test_mae}
    }
}

print("\n✓ Models and hyperparameters saved for retraining")

print("\n" + "="*60)
print("PERFORMANCE OPTIMIZATIONS APPLIED:")
print("="*60)
print("✅ Persisted train/test datasets once (MEMORY_AND_DISK)")
print("✅ Pre-assembled and persisted features for tree models")
print("✅ Avoided redundant .count() calls")
print(f"✅ Smart sampling: {SAMPLE_FRAC*100:.0f}% for fast iteration")
print("✅ Used best hyperparameters from previous CV tuning")
print("✅ GBT: maxDepth=12, stepSize=0.2")
print("✅ RF: numTrees=150, maxDepth=15")
print("="*60)

print(f"\n💡 To use more data:")
print("   - Change SAMPLE_FRAC to 0.10 (10%) for final training")
print("   - Change SAMPLE_FRAC to 1.0 for full dataset")
print(f"   - Current: {train_count:,} training rows")






[1/4] Training Simple Linear Regression (chip_power only)...
  Training   → R²: 0.5355 | RMSE: 1819.75 | MAE: 1197.88
  Test       → R²: 0.5704 | RMSE: 1696.50 | MAE: 1196.67
  Coefficient: 0.9728 | Intercept: 1619.17
  Training time: 4.12s

[2/4] Training Ridge Regression (all features, scaled)...
  Training   → R²: 0.6320 | RMSE: 1619.59 | MAE: 962.17
  Test       → R²: -0.6064 | RMSE: 3280.64 | MAE: 963.37
  Training time: 3.53s

[3/4] Training Gradient Boosted Trees (with best CV params)...
  Training   → R²: 0.7583 | RMSE: 1312.57 | MAE: 681.38
  Test       → R²: 0.8016 | RMSE: 1152.99 | MAE: 685.85
  Training time: 684.16s
  Hyperparameters: maxIter=30, maxDepth=12, stepSize=0.2, subsample=0.6

  Feature Importance:
    chip_power: 0.6501
    gpu_peak_power_node_watts: 0.0629
    peak_power_unit: 0.0496
    power_pct_max_index: 0.0448
    tensor_tflops_sm: 0.0324
    sm_clock: 0.0291
    vram_usage: 0.0209
    sm_active: 0.0207
    dram_active: 0.0153
    tensor_tflops: 0.0153
 

In [10]:
# # DIAGNOSTIC PLOTS: QQ Plot & Predicted vs Actual (ALL MODELS)
# # ========================================
# # Visualizes model performance and residual distribution for ALL trained models

# # Install visualization packages (SparkCaster kernel workaround)
# import subprocess
# import sys
# import importlib
# import site

# def install_and_import(package_name, import_name=None):
#     """Install package and reload site-packages to make it importable"""
#     if import_name is None:
#         import_name = package_name
    
#     try:
#         return __import__(import_name)
#     except ImportError:
#         print(f"Installing {package_name}...")
#         subprocess.check_call([
#             sys.executable, '-m', 'pip', 'install', 
#             '--user', '--upgrade', package_name
#         ])
#         # Reload site packages to pick up newly installed packages
#         importlib.reload(site)
#         # Force reimport of sys.path
#         import site as site_module
#         site_module.main()
#         return __import__(import_name)

# # Install required packages
# install_and_import('matplotlib')
# install_and_import('scipy')

# import matplotlib.pyplot as plt
# import numpy as np
# from scipy import stats

# print("="*60)
# print("GENERATING DIAGNOSTIC PLOTS FOR ALL MODELS")
# print("="*60)

# # ========================================
# # Sample data for visualization (use test set)
# # ========================================
# # Don't plot all rows - sample for performance
# PLOT_SAMPLE_SIZE = 10000  # Plot 10k points max

# # Define all models to plot
# models_to_plot = [
#     ('Simple LR (chip_power)', simple_lr_test_pred, simple_lr_test_r2),
#     ('Ridge (all features)', ridge_test_pred, ridge_test_r2),
#     ('Gradient Boosting', gbt_test_pred, gbt_test_r2),
#     ('Random Forest', rf_test_pred, rf_test_r2)
# ]

# # ========================================
# # Generate plots for each model
# # ========================================
# for model_name, test_pred_sdf, test_r2 in models_to_plot:
#     print(f"\n{'='*60}")
#     print(f"Model: {model_name} (Test R²: {test_r2:.4f})")
#     print('='*60)
    
#     # Sample and convert to pandas for plotting
#     test_sample = test_pred_sdf.select(target_col, "prediction") \
#         .sample(False, min(1.0, PLOT_SAMPLE_SIZE / test_count)) \
#         .toPandas()
    
#     actuals = test_sample[target_col].values
#     predictions = test_sample['prediction'].values
#     residuals = actuals - predictions
    
#     print(f"Plotting {len(actuals):,} sampled points from test set")
    
#     # Create filename-safe model name
#     filename_safe = model_name.replace(' ', '_').replace('(', '').replace(')', '').lower()
    
#     # ========================================
#     # Figure 1: QQ Plot (Residuals Normality Check)
#     # ========================================
#     fig1, ax1 = plt.subplots(figsize=(10, 6))
    
#     # Create QQ plot
#     stats.probplot(residuals, dist="norm", plot=ax1)
    
#     ax1.set_title(f'QQ Plot: Residuals Normality Check\n{model_name} (Test Set)', 
#                   fontsize=14, fontweight='bold')
#     ax1.set_xlabel('Theoretical Quantiles', fontsize=12)
#     ax1.set_ylabel('Sample Quantiles (Residuals)', fontsize=12)
#     ax1.grid(True, alpha=0.3)
    
#     # Style the points
#     ax1.get_lines()[0].set_markerfacecolor('blue')
#     ax1.get_lines()[0].set_markeredgecolor('blue')
#     ax1.get_lines()[0].set_markersize(4)
#     ax1.get_lines()[0].set_alpha(0.6)
    
#     # Calculate normality statistics
#     _, p_value = stats.shapiro(residuals[:5000] if len(residuals) > 5000 else residuals)
#     skewness = stats.skew(residuals)
#     kurtosis = stats.kurtosis(residuals)
    
#     # Add statistics text box
#     stats_text = f'Shapiro-Wilk p-value: {p_value:.4f}\n'
#     stats_text += f'Skewness: {skewness:.3f}\n'
#     stats_text += f'Kurtosis: {kurtosis:.3f}\n'
#     stats_text += f'\nResiduals ~ Normal' if p_value > 0.05 else f'\nResiduals not normal'
    
#     ax1.text(0.05, 0.95, stats_text,
#              transform=ax1.transAxes,
#              fontsize=10,
#              verticalalignment='top',
#              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
#     plt.tight_layout()
#     filename = f'qq_plot_{filename_safe}.png'
#     plt.savefig(filename, dpi=150, bbox_inches='tight')
#     print(f"  ✓ Saved: {filename}")
#     plt.show()
    
#     # ========================================
#     # Figure 2: Predicted vs Actual Scatter Plot
#     # ========================================
#     fig2, ax2 = plt.subplots(figsize=(10, 8))
    
#     # Scatter plot with alpha for overlap visualization
#     ax2.scatter(actuals, predictions, alpha=0.5, s=20, c='blue', edgecolors='none')
    
#     # Perfect prediction line (45-degree)
#     min_val = min(actuals.min(), predictions.min())
#     max_val = max(actuals.max(), predictions.max())
#     ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
#     # Add best fit line
#     z = np.polyfit(actuals, predictions, 1)
#     p = np.poly1d(z)
#     ax2.plot(actuals, p(actuals), "g-", linewidth=2, label=f'Best Fit: y={z[0]:.3f}x+{z[1]:.1f}')
    
#     ax2.set_xlabel(f'Actual {target_col}', fontsize=12)
#     ax2.set_ylabel(f'Predicted {target_col}', fontsize=12)
#     ax2.set_title(f'Predicted vs Actual: {model_name}\nTest Set (n={len(actuals):,})', 
#                   fontsize=14, fontweight='bold')
#     ax2.legend(loc='upper left', fontsize=10)
#     ax2.grid(True, alpha=0.3)
    
#     # Add metrics text box
#     metrics_text = f'Test R²: {test_r2:.4f}\n'
#     metrics_text += f'RMSE: {np.sqrt(np.mean(residuals**2)):.2f}\n'
#     metrics_text += f'MAE: {np.mean(np.abs(residuals)):.2f}\n'
#     metrics_text += f'Mean Residual: {np.mean(residuals):.2f}'
    
#     ax2.text(0.05, 0.95, metrics_text,
#              transform=ax2.transAxes,
#              fontsize=10,
#              verticalalignment='top',
#              bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
#     plt.tight_layout()
#     filename = f'predicted_vs_actual_{filename_safe}.png'
#     plt.savefig(filename, dpi=150, bbox_inches='tight')
#     print(f"  ✓ Saved: {filename}")
#     plt.show()
    
#     # ========================================
#     # Figure 3: Residuals Histogram
#     # ========================================
#     fig3, ax3 = plt.subplots(figsize=(10, 6))
    
#     ax3.hist(residuals, bins=50, alpha=0.7, color='blue', edgecolor='black')
#     ax3.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Residual')
#     ax3.axvline(np.mean(residuals), color='green', linestyle='--', linewidth=2, label=f'Mean: {np.mean(residuals):.2f}')
    
#     ax3.set_xlabel('Residuals (Actual - Predicted)', fontsize=12)
#     ax3.set_ylabel('Frequency', fontsize=12)
#     ax3.set_title(f'Residuals Distribution: {model_name}\nTest Set', 
#                   fontsize=14, fontweight='bold')
#     ax3.legend(fontsize=10)
#     ax3.grid(True, alpha=0.3, axis='y')
    
#     plt.tight_layout()
#     filename = f'residuals_histogram_{filename_safe}.png'
#     plt.savefig(filename, dpi=150, bbox_inches='tight')
#     print(f"  ✓ Saved: {filename}")
#     plt.show()
    
#     # Print diagnostic summary for this model
#     print(f"\n  Residuals Mean: {np.mean(residuals):.2f} (should be ~0)")
#     print(f"  Residuals Std: {np.std(residuals):.2f}")
#     print(f"  Residuals Range: [{residuals.min():.2f}, {residuals.max():.2f}]")
#     print(f"  Normality (Shapiro-Wilk p-value): {p_value:.4f}")
#     if p_value > 0.05:
#         print("    → Residuals appear normally distributed ✓")
#     else:
#         print("    → Residuals may not be normally distributed")

# print("\n" + "="*60)

# # ========================================
# # Generate Combined HTML Report
# # ========================================
# print("\n" + "="*60)
# print("GENERATING HTML REPORT")
# print("="*60)

# html_content = '''<!DOCTYPE html>
# <html>
# <head>
#     <title>Model Diagnostic Report - GPU Metrics Prediction</title>
#     <style>
#         body {
#             font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
#             margin: 40px;
#             background-color: #f5f5f5;
#         }
#         h1 {
#             color: #2c3e50;
#             border-bottom: 3px solid #3498db;
#             padding-bottom: 10px;
#         }
#         h2 {
#             color: #34495e;
#             margin-top: 40px;
#             background-color: #ecf0f1;
#             padding: 10px;
#             border-left: 5px solid #3498db;
#         }
#         .model-section {
#             background-color: white;
#             padding: 20px;
#             margin: 20px 0;
#             border-radius: 8px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#         }
#         .metrics {
#             display: grid;
#             grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
#             gap: 15px;
#             margin: 20px 0;
#         }
#         .metric-card {
#             background-color: #ecf0f1;
#             padding: 15px;
#             border-radius: 5px;
#             text-align: center;
#         }
#         .metric-label {
#             font-size: 12px;
#             color: #7f8c8d;
#             text-transform: uppercase;
#         }
#         .metric-value {
#             font-size: 24px;
#             font-weight: bold;
#             color: #2c3e50;
#             margin-top: 5px;
#         }
#         .plot-grid {
#             display: grid;
#             grid-template-columns: repeat(auto-fit, minmax(400px, 1fr));
#             gap: 20px;
#             margin: 20px 0;
#         }
#         .plot-container {
#             text-align: center;
#         }
#         .plot-container img {
#             max-width: 100%;
#             border: 1px solid #ddd;
#             border-radius: 5px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#         }
#         .plot-title {
#             font-weight: bold;
#             margin: 10px 0;
#             color: #34495e;
#         }
#         .best-model {
#             background-color: #d5f4e6;
#             border-left: 5px solid #27ae60;
#         }
#         .summary-table {
#             width: 100%;
#             border-collapse: collapse;
#             margin: 20px 0;
#             background-color: white;
#         }
#         .summary-table th {
#             background-color: #3498db;
#             color: white;
#             padding: 12px;
#             text-align: left;
#         }
#         .summary-table td {
#             padding: 10px;
#             border-bottom: 1px solid #ddd;
#         }
#         .summary-table tr:hover {
#             background-color: #f5f5f5;
#         }
#         .best-row {
#             background-color: #d5f4e6 !important;
#             font-weight: bold;
#         }
#     </style>
# </head>
# <body>
#     <h1>🔍 Model Diagnostic Report: GPU Metrics Prediction</h1>
#     <p><strong>Target Variable:</strong> redfish_power</p>
#     <p><strong>Features:</strong> tensor_util, gpu_util, tensor_tflops, chip_power</p>
#     <p><strong>Generated:</strong> ''' + time.strftime('%Y-%m-%d %H:%M:%S') + '''</p>
    
#     <h2>📊 Model Performance Summary</h2>
#     <table class="summary-table">
#         <thead>
#             <tr>
#                 <th>Model</th>
#                 <th>Test R²</th>
#                 <th>Test R²</th>
#                 <th>Test RMSE</th>
#                 <th>Test MAE</th>
#             </tr>
#         </thead>
#         <tbody>
# '''

# # Add each model's summary (using test R² since we no longer do CV)
# models_summary = [
#     ('Simple LR (chip_power)', simple_lr_test_r2, simple_lr_test_r2, simple_lr_test_rmse, simple_lr_test_mae),
#     ('Ridge (all features)', ridge_test_r2, ridge_test_r2, ridge_test_rmse, ridge_test_mae),
#     ('Gradient Boosting', gbt_test_r2, gbt_test_r2, gbt_test_rmse, gbt_test_mae),
#     ('Random Forest', rf_test_r2, rf_test_r2, rf_test_rmse, rf_test_mae)
# ]

# best_test_r2_value = max([m[1] for m in models_summary])

# for model_name, test_r2_display, test_r2, test_rmse, test_mae in models_summary:
#     row_class = 'best-row' if cv_r2 == best_cv_r2_value else ''
#     html_content += f'''            <tr class="{row_class}">
#                 <td>{model_name}</td>
#                 <td>{cv_r2:.4f}</td>
#                 <td>{test_r2:.4f}</td>
#                 <td>{test_rmse:.2f}</td>
#                 <td>{test_mae:.2f}</td>
#             </tr>
# '''

# html_content += '''        </tbody>
#     </table>
# '''

# # Add detailed sections for each model
# for model_name, test_pred_sdf, test_r2 in models_to_plot:
#     # Determine if this is the best model
#     is_best = test_r2 == max([m[2] for m in models_to_plot])
#     section_class = 'model-section best-model' if is_best else 'model-section'
#     best_badge = ' 🏆 BEST MODEL' if is_best else ''
    
#     # Create filename-safe model name
#     filename_safe = model_name.replace(' ', '_').replace('(', '').replace(')', '').lower()
    
#     # Get model-specific metrics
#     if model_name == 'Simple LR (chip_power)':
#         cv_r2, test_rmse, test_mae = simple_lr_cv_r2, simple_lr_test_rmse, simple_lr_test_mae
#     elif model_name == 'Ridge (all features)':
#         cv_r2, test_rmse, test_mae = ridge_cv_r2, ridge_test_rmse, ridge_test_mae
#     elif model_name == 'Gradient Boosting':
#         cv_r2, test_rmse, test_mae = gbt_cv_r2, gbt_test_rmse, gbt_test_mae
#     else:  # Random Forest
#         cv_r2, test_rmse, test_mae = rf_cv_r2, rf_test_rmse, rf_test_mae
    
#     html_content += f'''
#     <div class="{section_class}">
#         <h2>{model_name}{best_badge}</h2>
        
#         <div class="metrics">
#             <div class="metric-card">
#                 <div class="metric-label">CV R² (5-fold)</div>
#                 <div class="metric-value">{cv_r2:.4f}</div>
#             </div>
#             <div class="metric-card">
#                 <div class="metric-label">Test R²</div>
#                 <div class="metric-value">{test_r2:.4f}</div>
#             </div>
#             <div class="metric-card">
#                 <div class="metric-label">Test RMSE</div>
#                 <div class="metric-value">{test_rmse:.2f}</div>
#             </div>
#             <div class="metric-card">
#                 <div class="metric-label">Test MAE</div>
#                 <div class="metric-value">{test_mae:.2f}</div>
#             </div>
#         </div>
        
#         <div class="plot-grid">
#             <div class="plot-container">
#                 <div class="plot-title">QQ Plot (Residuals Normality)</div>
#                 <img src="qq_plot_{filename_safe}.png" alt="QQ Plot">
#             </div>
#             <div class="plot-container">
#                 <div class="plot-title">Predicted vs Actual</div>
#                 <img src="predicted_vs_actual_{filename_safe}.png" alt="Predicted vs Actual">
#             </div>
#             <div class="plot-container">
#                 <div class="plot-title">Residuals Distribution</div>
#                 <img src="residuals_histogram_{filename_safe}.png" alt="Residuals Histogram">
#             </div>
#         </div>
#     </div>
# '''

# html_content += '''
#     <div class="model-section">
#         <h2>📝 Notes</h2>
#         <ul>
#             <li><strong>CV R²:</strong> Average R² across 5-fold cross-validation (training performance estimate)</li>
#             <li><strong>Test R²:</strong> R² on held-out test set (true generalization performance)</li>
#             <li><strong>QQ Plot:</strong> Points should follow the red line if residuals are normally distributed</li>
#             <li><strong>Predicted vs Actual:</strong> Points should cluster around the red diagonal line</li>
#             <li><strong>Residuals:</strong> Should be centered around zero with symmetric distribution</li>
#         </ul>
#     </div>
# </body>
# </html>
# '''

# # Write HTML file
# html_filename = 'model_diagnostics_report.html'
# with open(html_filename, 'w') as f:
#     f.write(html_content)

# print(f"\n✓ Saved HTML report: {html_filename}")
# print(f"  Open this file in a web browser to view all diagnostics")

# print("ALL DIAGNOSTIC PLOTS COMPLETE")
# print("="*60)
# print(f"\n✓ Generated 12 plots total (3 plots × 4 models)")
# print(f"✓ Files saved in current directory")
# print("\nFiles created:")
# print("  - qq_plot_simple_lr_chip_power.png")
# print("  - predicted_vs_actual_simple_lr_chip_power.png")
# print("  - residuals_histogram_simple_lr_chip_power.png")
# print("  - qq_plot_ridge_all_features.png")
# print("  - predicted_vs_actual_ridge_all_features.png")
# print("  - residuals_histogram_ridge_all_features.png")
# print("  - qq_plot_gradient_boosting.png")
# print("  - predicted_vs_actual_gradient_boosting.png")
# print("  - residuals_histogram_gradient_boosting.png")
# print("  - qq_plot_random_forest.png")
# print("  - predicted_vs_actual_random_forest.png")
# print("  - residuals_histogram_random_forest.png")
# print("="*60)



In [11]:
# # Unpersist to free memory
assembled_train.unpersist()
assembled_test.unpersist()
#assembled_train_scaled.unpersist()
#assembled_test_scaled.unpersist()
train_sdf.unpersist()
test_sdf.unpersist()

#spark.catalog.clearCache()
print("\n✓ Training complete and memory released")


✓ Training complete and memory released


In [12]:
# FINAL TRAINING: Retrain Best Model on 10% Sample
# ========================================
# Takes the best model from Cell 8 and retrains on larger dataset
# Dynamically uses the hyperparameters from the best model

import time
from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler as MLStandardScaler
from pyspark.ml.regression import LinearRegression as MLLinearRegression, GBTRegressor, RandomForestRegressor as MLRandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F

print("="*60)
print("FINAL TRAINING ON 10% SAMPLE")
print("="*60)

# ========================================
# CONFIGURATION - Larger Sample
# ========================================
FINAL_SAMPLE_FRAC = 0.10  # 10% sample for final training
SEED = 42

feature_cols = [c for c in num_cols if c != "redfish_power"]
target_col = "redfish_power"

# Get best model info from Cell 8
best_model_info = trained_models[best_model_name]
best_hyperparams = best_model_info['hyperparameters']

print(f"\nBest Model from Cell 8: {best_model_name}")
print(f"Test R²: {best_model_info['metrics']['test_r2']:.4f}")
print(f"Hyperparameters: {best_hyperparams}")
print(f"Increasing sample from {SAMPLE_FRAC*100:.0f}% to {FINAL_SAMPLE_FRAC*100:.0f}%...")

# ========================================
# Data Preparation - 10% Sample
# ========================================
print("\n[1/4] Preparing 10% sample...")

# Keep only necessary columns early (filter out nulls in target)
base_labeled_final = sdf_num.select(*(feature_cols + [target_col])) \
    .where(F.col(target_col).isNotNull())

# Add row ID for deterministic split
base_labeled_final = base_labeled_final.withColumn("_split_id", F.monotonically_increasing_id())

# Split FIRST, then sample
train_sdf_final, test_sdf_final = base_labeled_final.randomSplit([0.7, 0.3], seed=SEED)

# Sample after split
train_sdf_final = train_sdf_final.sample(False, FINAL_SAMPLE_FRAC, SEED)
test_sdf_final = test_sdf_final.sample(False, FINAL_SAMPLE_FRAC, SEED + 1)

# Drop split ID
train_sdf_final = train_sdf_final.drop("_split_id")
test_sdf_final = test_sdf_final.drop("_split_id")

# Cache and materialize
train_sdf_final.cache()
test_sdf_final.cache()

train_count_final = train_sdf_final.count()
test_count_final = test_sdf_final.count()

print(f"Training set: {train_count_final:,} rows (cached)")
print(f"Test set: {test_count_final:,} rows (cached)")

# ========================================
# Feature Assembly
# ========================================
print("\n[2/4] Assembling features...")

assembler_final = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)

assembled_train_final = assembler_final.transform(train_sdf_final) \
    .select("features_raw", target_col)
assembled_train_final.cache()

assembled_test_final = assembler_final.transform(test_sdf_final) \
    .select("features_raw", target_col)
assembled_test_final.cache()

_ = assembled_train_final.count()
_ = assembled_test_final.count()

print("✓ Features assembled and cached")

# ========================================
# Retrain Best Model with Dynamic Hyperparameters
# ========================================
print(f"\n[3/4] Retraining {best_model_name} on 10% sample...")

# Create evaluators
evaluator_r2_final = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")
evaluator_rmse_final = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
evaluator_mae_final = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="mae")

# Dynamically create model based on best_model_name with saved hyperparameters
if best_model_name == 'Simple LR (chip_power)':
    assembler_chip_final = VectorAssembler(
        inputCols=["chip_power"],
        outputCol="features_chip"
    )
    
    simple_lr_final = MLLinearRegression(
        featuresCol="features_chip",
        labelCol=target_col,
        predictionCol="prediction",
        maxIter=100,
        standardization=False
    )
    
    pipeline_final = Pipeline(stages=[assembler_chip_final, simple_lr_final])
    train_data_final = train_sdf_final
    test_data_final = test_sdf_final

elif best_model_name == 'Ridge (all features)':
    scaler_final = MLStandardScaler(
        inputCol="features_raw",
        outputCol="features_scaled",
        withStd=True,
        withMean=False
    )
    
    ridge_lr_final = MLLinearRegression(
        featuresCol="features_raw",
        labelCol=target_col,
        predictionCol="prediction",
        maxIter=100,
        regParam=best_hyperparams['regParam'],  # ✓ Dynamic
        elasticNetParam=best_hyperparams['elasticNetParam'],  # ✓ Dynamic
        standardization=False
    )
    
    pipeline_final = Pipeline(stages=[scaler_final, ridge_lr_final])
    train_data_final = assembled_train_final
    test_data_final = assembled_test_final

elif best_model_name == 'Gradient Boosting':
    gbt_final = GBTRegressor(
        featuresCol="features_raw",
        labelCol=target_col,
        predictionCol="prediction",
        maxIter=best_hyperparams['maxIter'],  # ✓ Dynamic
        maxDepth=best_hyperparams['maxDepth'],  # ✓ Dynamic
        stepSize=best_hyperparams['stepSize'],  # ✓ Dynamic
        subsamplingRate=best_hyperparams['subsamplingRate'],  # ✓ Dynamic
        minInstancesPerNode=best_hyperparams['minInstancesPerNode'],  # ✓ Dynamic
        maxBins=best_hyperparams['maxBins'],  # ✓ Dynamic
        seed=SEED
    )
    model_final = gbt_final
    train_data_final = assembled_train_final
    test_data_final = assembled_test_final

else:  # Random Forest
    rf_final = MLRandomForestRegressor(
        featuresCol="features_raw",
        labelCol=target_col,
        predictionCol="prediction",
        numTrees=best_hyperparams['numTrees'],  # ✓ Dynamic
        maxDepth=best_hyperparams['maxDepth'],  # ✓ Dynamic
        featureSubsetStrategy=best_hyperparams['featureSubsetStrategy'],  # ✓ Dynamic
        minInstancesPerNode=best_hyperparams['minInstancesPerNode'],  # ✓ Dynamic
        subsamplingRate=best_hyperparams['subsamplingRate'],  # ✓ Dynamic
        seed=SEED
    )
    model_final = rf_final
    train_data_final = assembled_train_final
    test_data_final = assembled_test_final

# Train (no CV, just direct training)
final_start = time.time()

if best_model_name in ['Simple LR (chip_power)', 'Ridge (all features)']:
    final_model = pipeline_final.fit(train_data_final)
else:
    final_model = model_final.fit(train_data_final)

final_time = time.time() - final_start

# Evaluate on train/test
final_train_pred = final_model.transform(train_data_final)
final_test_pred = final_model.transform(test_data_final)

final_train_r2 = evaluator_r2_final.evaluate(final_train_pred)
final_train_rmse = evaluator_rmse_final.evaluate(final_train_pred)
final_train_mae = evaluator_mae_final.evaluate(final_train_pred)
final_test_r2 = evaluator_r2_final.evaluate(final_test_pred)
final_test_rmse = evaluator_rmse_final.evaluate(final_test_pred)
final_test_mae = evaluator_mae_final.evaluate(final_test_pred)

print(f"\n  Training   → R²: {final_train_r2:.4f} | RMSE: {final_train_rmse:.2f} | MAE: {final_train_mae:.2f}")
print(f"  Test       → R²: {final_test_r2:.4f} | RMSE: {final_test_rmse:.2f} | MAE: {final_test_mae:.2f}")
print(f"  Training time: {final_time:.2f}s")

# Show feature importance for tree models
if best_model_name in ['Gradient Boosting', 'Random Forest']:
    feature_importance_final = list(zip(feature_cols, final_model.featureImportances.toArray()))
    feature_importance_final.sort(key=lambda x: x[1], reverse=True)
    print(f"\n  Feature Importance:")
    for feat, imp in feature_importance_final:
        print(f"    {feat}: {imp:.4f}")

# ========================================
# Comparison with 2% Sample
# ========================================
print("\n[4/4] Comparison: {:.0f}% vs {:.0f}% Sample".format(SAMPLE_FRAC*100, FINAL_SAMPLE_FRAC*100))
print("="*60)

import pandas as pd
comparison_final = pd.DataFrame({
    'Metric': ['Sample Size', 'Training Rows', 'Test Rows', 'Test R²', 'Test RMSE', 'Test MAE', 'Training Time (s)'],
    f'{SAMPLE_FRAC*100:.0f}% Sample': [
        f'{SAMPLE_FRAC*100:.0f}%',
        f'{train_count:,}',
        f'{test_count:,}',
        f'{best_model_info["metrics"]["test_r2"]:.4f}',
        f'{best_model_info["metrics"]["test_rmse"]:.2f}',
        f'{best_model_info["metrics"]["test_mae"]:.2f}',
        'N/A'
    ],
    f'{FINAL_SAMPLE_FRAC*100:.0f}% Sample': [
        f'{FINAL_SAMPLE_FRAC*100:.0f}%',
        f'{train_count_final:,}',
        f'{test_count_final:,}',
        f'{final_test_r2:.4f}',
        f'{final_test_rmse:.2f}',
        f'{final_test_mae:.2f}',
        f'{final_time:.2f}'
    ]
})
print(comparison_final.to_string(index=False))

improvement = final_test_r2 - best_model_info['metrics']['test_r2']
print(f"\n✓ Test R² improvement: {improvement:+.4f} ({improvement/best_model_info['metrics']['test_r2']*100:+.1f}%)")

# ========================================
# Save Final Model Summary
# ========================================
print("\n" + "="*60)
print("FINAL MODEL SUMMARY")
print("="*60)
print(f"Model: {best_model_name}")
print(f"Training Data: {train_count_final:,} rows ({FINAL_SAMPLE_FRAC*100:.0f}% of labeled data)")
print(f"Hyperparameters: {best_hyperparams}")
print(f"Test R²: {final_test_r2:.4f}")
print(f"Test RMSE: {final_test_rmse:.2f}")
print(f"Test MAE: {final_test_mae:.2f}")
print(f"Overfit Gap: {final_train_r2 - final_test_r2:.4f}")
print("="*60)

print("\n💾 To save this model for production use:")
print("   final_model.write().overwrite().save('/path/to/model')")

# Clean up
assembled_train_final.unpersist()
assembled_test_final.unpersist()
train_sdf_final.unpersist()
test_sdf_final.unpersist()

print("\n✓ Final training complete and memory released")


FINAL TRAINING ON 10% SAMPLE

Best Model from Cell 8: Gradient Boosting
Test R²: 0.8016
Hyperparameters: {'maxIter': 30, 'maxDepth': 12, 'stepSize': 0.2, 'subsamplingRate': 0.6, 'minInstancesPerNode': 100, 'maxBins': 32}
Increasing sample from 2% to 10%...

[1/4] Preparing 10% sample...
Training set: 54,407,093 rows (cached)
Test set: 23,316,046 rows (cached)

[2/4] Assembling features...
✓ Features assembled and cached

[3/4] Retraining Gradient Boosting on 10% sample...

  Training   → R²: 0.7829 | RMSE: 1224.01 | MAE: 677.72
  Test       → R²: 0.7270 | RMSE: 1423.49 | MAE: 679.35
  Training time: 1224.39s

  Feature Importance:
    chip_power: 0.6717
    gpu_peak_power_node_watts: 0.0620
    peak_power_unit: 0.0520
    power_pct_max_index: 0.0468
    tensor_tflops_sm: 0.0422
    vram_usage: 0.0207
    sm_active: 0.0200
    dram_active: 0.0145
    tensor_tflops: 0.0108
    sm_occupancy: 0.0107
    sm_clock: 0.0088
    gpu_util: 0.0081
    peak_tflops_unit: 0.0081
    compute_to_memor

In [13]:
# IMPUTE MISSING VALUES
# ========================================
# Apply the trained model to impute missing redfish_power values
# Add additional features here as needed

import time
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler

print("="*60)
print("IMPUTATION PIPELINE")
print("="*60)

print(f"\nModel: {best_model_name}")
print(f"Model hyperparameters: {best_hyperparams}")

# Step 1: Check original data
print("\n[1/5] Checking original data...")
total_rows = df.count()
null_count_before = df.filter(F.col(target_col).isNull()).count()
print(f"Total rows: {total_rows:,}")
print(f"Rows with NULL {target_col}: {null_count_before:,} ({100*null_count_before/total_rows:.1f}%)")

# Step 2: Assemble features for prediction
print("\n[2/5] Assembling features...")
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="keep"  # Keep rows with null features, we'll handle them
)

df_with_features = assembler.transform(df)
print("✓ Features assembled")

# Step 3: Apply model to generate predictions
print("\n[3/5] Generating predictions for all rows...")
print("Note: Model will predict for all rows, but we'll only use predictions where redfish_power is NULL")

impute_start = time.time()
df_with_predictions = final_model.transform(df_with_features)
impute_time = time.time() - impute_start
print(f"✓ Predictions generated in {impute_time:.1f}s")

# Step 4: Create final imputed column - use prediction only where original is NULL
print("\n[4/5] Creating imputed dataset...")
df_imputed = df_with_predictions.withColumn(
    target_col,
    F.when(F.col(target_col).isNull(), F.col("prediction"))
      .otherwise(F.col(target_col))
)

# Clean up: drop temporary columns
columns_to_drop = ["features_raw", "prediction"]
for col in columns_to_drop:
    if col in df_imputed.columns:
        df_imputed = df_imputed.drop(col)

print("✓ Imputation complete")

# Verify imputation
# null_count_after = df_imputed.filter(F.col(target_col).isNull()).count()
# imputed_count = null_count_before - null_count_after
# print(f"\nImputation results:")
# print(f"  Original NULLs: {null_count_before:,}")
# print(f"  Remaining NULLs: {null_count_after:,}")
# print(f"  Successfully imputed: {imputed_count:,}")

# if null_count_after > 0:
#     print(f"  ⚠️  {null_count_after:,} rows still NULL (likely had NULL features)")

# ========================================
# ADD ADDITIONAL FEATURES HERE
# ========================================
# Example:
# print("\n[5/5] Adding additional features...")
# df_imputed = df_imputed.withColumn("new_feature", F.col("existing_col") * 2)
# print("✓ Additional features added")

print("\n[5/5] Feature engineering...")
# Add your custom features here
# df_imputed = df_imputed.withColumn("power_efficiency", F.col("redfish_power") / F.col("chip_power"))
# df_imputed = df_imputed.withColumn("imputed_flag", F.when(F.col(target_col).isNull(), 1).otherwise(0))
print("✓ No additional features configured (add them above)")

# Show final schema
print(f"\nFinal schema: {len(df_imputed.columns)} columns")
print("Columns:", df_imputed.columns)

# Show sample
print("\nSample imputed data:")
df_imputed.select("datestamp", "node", target_col, "chip_power").show(10)

# Cache for write step
df_imputed.cache()
imputed_row_count = df_imputed.count()
#print(f"\n✓ Imputed dataset cached: {imputed_row_count:,} rows")

print("\n" + "="*60)
print("IMPUTATION COMPLETE")
print("="*60)
print("\nNext: Run the WRITE TO NESSIE cell to save results")


IMPUTATION PIPELINE

Model: Gradient Boosting
Model hyperparameters: {'maxIter': 30, 'maxDepth': 12, 'stepSize': 0.2, 'subsamplingRate': 0.6, 'minInstancesPerNode': 100, 'maxBins': 32}

[1/5] Checking original data...
Total rows: 1,118,490,310
Rows with NULL redfish_power: 320,031,736 (28.6%)

[2/5] Assembling features...
✓ Features assembled

[3/5] Generating predictions for all rows...
Note: Model will predict for all rows, but we'll only use predictions where redfish_power is NULL
✓ Predictions generated in 0.2s

[4/5] Creating imputed dataset...
✓ Imputation complete

[5/5] Feature engineering...
✓ No additional features configured (add them above)

Final schema: 46 columns
Columns: ['datestamp', 'node', 'gpu_util', 'tensor_util', 'chip_power', 'dram_active', 'mem_copy_util', 'vram_usage', 'sm_active', 'sm_clock', 'sm_occupancy', 'region', 'zone', 'cluster', 'cluster_org', 'cw_sku', 'model', 'serial', 'redfish_power', 'customer_name', 'flag_is_coreweave', 'model_imputed', 'product'

In [14]:
df_imputed.printSchema()

# ========================================
# Clean up training data cache
# ========================================
# sdf_num is no longer needed after imputation (Cell 13)
# Free memory before classification analysis (Cell 15)
print("\n" + "="*60)
print("CLEANING UP TRAINING CACHE")
print("="*60)

sdf_num.unpersist()
print("✓ Unpersisted sdf_num to free memory")
print("  df_imputed remains cached for classification and Nessie write")

# Note: train_df/test_df from Cell 12 are already unpersisted at end of Cell 12
# Note: df_imputed is still cached for use in Cell 15 (classification) and Cell 17 (Nessie write)

print("="*60)
print("\n✓ Model training complete")
print("\nNext: Run classification and visualization cells to generate plots")


root
 |-- datestamp: timestamp (nullable = true)
 |-- node: string (nullable = true)
 |-- gpu_util: double (nullable = true)
 |-- tensor_util: double (nullable = true)
 |-- chip_power: double (nullable = true)
 |-- dram_active: double (nullable = true)
 |-- mem_copy_util: double (nullable = true)
 |-- vram_usage: double (nullable = true)
 |-- sm_active: double (nullable = true)
 |-- sm_clock: double (nullable = true)
 |-- sm_occupancy: double (nullable = true)
 |-- region: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- cluster: string (nullable = true)
 |-- cluster_org: string (nullable = true)
 |-- cw_sku: string (nullable = true)
 |-- model: string (nullable = true)
 |-- serial: string (nullable = true)
 |-- redfish_power: double (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- flag_is_coreweave: string (nullable = true)
 |-- model_imputed: string (nullable = true)
 |-- product: string (nullable = true)
 |-- product_resolved: string (nullable = t

In [15]:
# ========================================
# VALIDATION: region_rollup and gen in imputed data
# ========================================

print("="*80)
print("VALIDATION CHECKS FOR NEW FEATURES")
print("="*80)

# 1. Schema Validation
print("\n[1/4] Schema Validation - Checking df_imputed contains new columns...")
imputed_columns = df_imputed.columns
if "region_rollup" in imputed_columns:
    print("✓ region_rollup exists in df_imputed schema")
else:
    print("✗ ERROR: region_rollup NOT found in df_imputed schema")
    
if "gen" in imputed_columns:
    print("✓ gen exists in df_imputed schema")
else:
    print("✗ ERROR: gen NOT found in df_imputed schema")

# 2. Data Sanity - region_rollup mapping
print("\n[2/4] Data Sanity - Checking region to region_rollup mapping...")
print("\nDistinct region -> region_rollup mappings:")
df_imputed.select("region", "region_rollup").distinct().orderBy("region").show(50, truncate=False)

# Verify logic: EU regions -> EU, others -> NAM
eu_regions = df_imputed.filter(F.col("region").startswith("EU")).select("region_rollup").distinct().collect()
non_eu_regions = df_imputed.filter(~F.col("region").startswith("EU")).select("region_rollup").distinct().collect()

print("\nValidation checks:")
if all(row.region_rollup == "EU" for row in eu_regions):
    print("✓ All EU-prefixed regions map to 'EU'")
else:
    print("✗ ERROR: Some EU-prefixed regions do not map to 'EU'")
    
if all(row.region_rollup == "NAM" for row in non_eu_regions):
    print("✓ All non-EU regions map to 'NAM'")
else:
    print("✗ ERROR: Some non-EU regions do not map to 'NAM'")

# 3. gen distribution
print("\n[3/4] Data Sanity - Checking gen distribution...")
print("\ngen distribution:")
df_imputed.groupBy("gen").count().orderBy("gen").show()

# Sample with both features
print("\n[4/4] Sample data with new features...")
print("\nSample showing region, region_rollup, product_resolved, gen:")
df_imputed.select(
    "region", "region_rollup", "product_resolved", "gen", 
    "customer_segment", "is_training"
).show(20, truncate=False)

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)


VALIDATION CHECKS FOR NEW FEATURES

[1/4] Schema Validation - Checking df_imputed contains new columns...
✓ region_rollup exists in df_imputed schema
✓ gen exists in df_imputed schema

[2/4] Data Sanity - Checking region to region_rollup mapping...

Distinct region -> region_rollup mappings:
+-------------+-------------+
|region       |region_rollup|
+-------------+-------------+
|NULL         |NAM          |
|CA-EAST-01   |NAM          |
|EU-NORTH-02  |EU           |
|EU-NORTH-04  |EU           |
|EU-NORTH-05  |EU           |
|EU-SOUTH-03  |EU           |
|EU-SOUTH-04  |EU           |
|LAS1         |NAM          |
|ORD1         |NAM          |
|RDU1         |NAM          |
|RNO2         |NAM          |
|US-CENTRAL-01|NAM          |
|US-CENTRAL-02|NAM          |
|US-CENTRAL-03|NAM          |
|US-CENTRAL-04|NAM          |
|US-CENTRAL-05|NAM          |
|US-CENTRAL-06|NAM          |
|US-CENTRAL-07|NAM          |
|US-CENTRAL-08|NAM          |
|US-CENTRAL-09|NAM          |
|US-CENTRAL-10|NA

In [16]:
summary_sql = '''
WITH base AS (
    SELECT
        datestamp,
        DATE_TRUNC('day', datestamp) AS day,
        region,
        zone,
        trim(customer_segment) AS customer_segment,
        trim(customer_name)    AS customer_name,
        coalesce(trim(cast(is_training  AS string)), 'unknown') AS is_training,
        coalesce(trim(cast(is_multinode AS string)), 'unknown') AS is_multinode,
        trim(product_resolved) AS product_resolved,
        trim(product_segment)  AS product_segment,
        cast(gpu_count_expected AS int) AS gpu_count_expected,
        gen,
        region_rollup,
        tensor_tflops,
        node,
        peak_power_unit,
        gpu_util,
        tensor_util,
        chip_power,
        redfish_power,
        dram_active,
        mem_copy_util,
        vram_usage,
        sm_active,
        sm_clock,
        sm_occupancy,
        memory_boundness_index,
        TFLOPS_per_watt_efficiency,
        compute_occupancy_index,
        tensor_util_dram_index,
        tensor_tflops_sm,
        tensor_to_gpu_ratio_index,
        memory_intensity_index,
        power_pct_max_index,
        compute_to_memory_ratio_index,
        compute_pct_max_index
    FROM {raw_imputed_ref}
),

per_ts AS (
    SELECT
        day,
        datestamp AS ts15,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        SUM(tensor_tflops) AS tensor_tflops_fleet_inst,
        SUM(tensor_tflops_sm) AS tensor_tflops_sm_fleet_inst,
        SUM(chip_power) AS chip_power_fleet_inst,
        SUM(redfish_power) AS redfish_power_fleet_inst,
        SUM(vram_usage) AS vram_usage_fleet_inst,
        COUNT(DISTINCT node) AS node_count
    FROM base
    GROUP BY
        day,
        datestamp,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name
),

fleet_daily AS (
    SELECT
        day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        tensor_tflops_fleet_avg,
        element_at(tensor_tflops_fleet_pct, 1) AS tensor_tflops_fleet_p50,
        element_at(tensor_tflops_fleet_pct, 2) AS tensor_tflops_fleet_p95,
        element_at(tensor_tflops_fleet_pct, 3) AS tensor_tflops_fleet_p99,
        element_at(tensor_tflops_sm_fleet_pct, 1) AS tensor_tflops_sm_fleet_p50,
        element_at(tensor_tflops_sm_fleet_pct, 2) AS tensor_tflops_sm_fleet_p95,
        element_at(tensor_tflops_sm_fleet_pct, 3) AS tensor_tflops_sm_fleet_p99,
        element_at(chip_power_fleet_pct, 1) AS chip_power_fleet_p50,
        element_at(chip_power_fleet_pct, 2) AS chip_power_fleet_p95,
        element_at(chip_power_fleet_pct, 3) AS chip_power_fleet_p99,
        element_at(redfish_power_fleet_pct, 1) AS redfish_power_fleet_p50,
        element_at(redfish_power_fleet_pct, 2) AS redfish_power_fleet_p95,
        element_at(redfish_power_fleet_pct, 3) AS redfish_power_fleet_p99,
        element_at(vram_usage_fleet_pct, 1) AS vram_usage_fleet_p50,
        element_at(vram_usage_fleet_pct, 2) AS vram_usage_fleet_p95,
        element_at(vram_usage_fleet_pct, 3) AS vram_usage_fleet_p99
    FROM (
        SELECT
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name,
            SUM(tensor_tflops_fleet_inst) / 96.0 AS tensor_tflops_fleet_avg,
            percentile_approx(tensor_tflops_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS tensor_tflops_fleet_pct,
            percentile_approx(tensor_tflops_sm_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS tensor_tflops_sm_fleet_pct,
            percentile_approx(chip_power_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS chip_power_fleet_pct,
            percentile_approx(redfish_power_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS redfish_power_fleet_pct,
            percentile_approx(vram_usage_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS vram_usage_fleet_pct
        FROM per_ts
        GROUP BY
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name
    ) t
),

node_daily AS (
    SELECT
        day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        AVG(node_count) AS node_count_daily_avg
    FROM per_ts
    GROUP BY
        day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name
),

-- =====================================================================
-- METHOD 2 (bounded utilization ratios only): average each node to its
-- DAILY MEAN first, THEN percentile across node-day means. Matches the
-- grain capacity is actually reclaimed at and removes the 15-min bimodal
-- (idle~0 / busy~1) spikes that make raw-grain p50/p95 degenerate.
-- Applies to: gpu_util, tensor_util, dram_active, mem_copy_util,
--             sm_active, sm_occupancy.
-- (power_pct_max and compute_pct_max are intentionally NOT here -- they
--  stay on raw-grain percentiles because peak is the physical constraint.)
-- =====================================================================
node_util_daily AS (
    SELECT
        day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        node,
        AVG(LEAST(gpu_util, 1))      AS gpu_util_node,
        AVG(LEAST(tensor_util, 1))   AS tensor_util_node,
        AVG(LEAST(dram_active, 1))   AS dram_active_node,
        AVG(LEAST(mem_copy_util, 1)) AS mem_copy_util_node,
        AVG(LEAST(sm_active, 1))     AS sm_active_node,
        AVG(LEAST(sm_occupancy, 1))  AS sm_occupancy_node
    FROM base
    GROUP BY
        day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        node
),

util_daily AS (
    SELECT
        day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        element_at(gpu_util_pct, 1)      AS gpu_util_p50,
        element_at(gpu_util_pct, 2)      AS gpu_util_p95,
        element_at(gpu_util_pct, 3)      AS gpu_util_p99,
        element_at(tensor_util_pct, 1)   AS tensor_util_p50,
        element_at(tensor_util_pct, 2)   AS tensor_util_p95,
        element_at(tensor_util_pct, 3)   AS tensor_util_p99,
        element_at(dram_active_pct, 1)   AS dram_active_p50,
        element_at(dram_active_pct, 2)   AS dram_active_p95,
        element_at(dram_active_pct, 3)   AS dram_active_p99,
        element_at(mem_copy_util_pct, 1) AS mem_copy_util_p50,
        element_at(mem_copy_util_pct, 2) AS mem_copy_util_p95,
        element_at(mem_copy_util_pct, 3) AS mem_copy_util_p99,
        element_at(sm_active_pct, 1)     AS sm_active_p50,
        element_at(sm_active_pct, 2)     AS sm_active_p95,
        element_at(sm_active_pct, 3)     AS sm_active_p99,
        element_at(sm_occupancy_pct, 1)  AS sm_occupancy_p50,
        element_at(sm_occupancy_pct, 2)  AS sm_occupancy_p95,
        element_at(sm_occupancy_pct, 3)  AS sm_occupancy_p99
    FROM (
        SELECT
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name,
            percentile_approx(gpu_util_node,      array(0.50, 0.95, 0.99), 1000) AS gpu_util_pct,
            percentile_approx(tensor_util_node,   array(0.50, 0.95, 0.99), 1000) AS tensor_util_pct,
            percentile_approx(dram_active_node,   array(0.50, 0.95, 0.99), 1000) AS dram_active_pct,
            percentile_approx(mem_copy_util_node, array(0.50, 0.95, 0.99), 1000) AS mem_copy_util_pct,
            percentile_approx(sm_active_node,     array(0.50, 0.95, 0.99), 1000) AS sm_active_pct,
            percentile_approx(sm_occupancy_node,  array(0.50, 0.95, 0.99), 1000) AS sm_occupancy_pct
        FROM node_util_daily
        GROUP BY
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name
    ) u
),

final_rowstats AS (
    SELECT
        day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        peak_power_unit,
        element_at(sm_clock_pct, 1) AS sm_clock_p50,
        element_at(sm_clock_pct, 2) AS sm_clock_p95,
        element_at(sm_clock_pct, 3) AS sm_clock_p99,
        element_at(memory_boundness_index_pct, 1) AS memory_boundness_index_p50,
        element_at(memory_boundness_index_pct, 2) AS memory_boundness_index_p95,
        element_at(memory_boundness_index_pct, 3) AS memory_boundness_index_p99,
        element_at(TFLOPS_per_watt_efficiency_pct, 1) AS TFLOPS_per_watt_efficiency_p50,
        element_at(TFLOPS_per_watt_efficiency_pct, 2) AS TFLOPS_per_watt_efficiency_p95,
        element_at(TFLOPS_per_watt_efficiency_pct, 3) AS TFLOPS_per_watt_efficiency_p99,
        element_at(compute_occupancy_index_pct, 1) AS compute_occupancy_index_p50,
        element_at(compute_occupancy_index_pct, 2) AS compute_occupancy_index_p95,
        element_at(compute_occupancy_index_pct, 3) AS compute_occupancy_index_p99,
        element_at(tensor_util_dram_index_pct, 1) AS tensor_util_dram_index_p50,
        element_at(tensor_util_dram_index_pct, 2) AS tensor_util_dram_index_p95,
        element_at(tensor_util_dram_index_pct, 3) AS tensor_util_dram_index_p99,
        element_at(tensor_to_gpu_ratio_pct, 1) AS tensor_to_gpu_ratio_p50,
        element_at(tensor_to_gpu_ratio_pct, 2) AS tensor_to_gpu_ratio_p95,
        element_at(tensor_to_gpu_ratio_pct, 3) AS tensor_to_gpu_ratio_p99,
        element_at(memory_intensity_pct, 1) AS memory_intensity_p50,
        element_at(memory_intensity_pct, 2) AS memory_intensity_p95,
        element_at(memory_intensity_pct, 3) AS memory_intensity_p99,
        element_at(power_pct_max_pct, 1) AS power_pct_max_p50,
        element_at(power_pct_max_pct, 2) AS power_pct_max_p95,
        element_at(power_pct_max_pct, 3) AS power_pct_max_p99,
        element_at(compute_to_memory_ratio_pct, 1) AS compute_to_memory_ratio_p50,
        element_at(compute_to_memory_ratio_pct, 2) AS compute_to_memory_ratio_p95,
        element_at(compute_to_memory_ratio_pct, 3) AS compute_to_memory_ratio_p99,
        element_at(compute_pct_max_pct, 1) AS compute_pct_max_p50,
        element_at(compute_pct_max_pct, 2) AS compute_pct_max_p95,
        element_at(compute_pct_max_pct, 3) AS compute_pct_max_p99
    FROM (
        SELECT
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name,
            AVG(peak_power_unit) AS peak_power_unit,
            percentile_approx(sm_clock, array(0.50, 0.95, 0.99), 1000) AS sm_clock_pct,
            percentile_approx(memory_boundness_index, array(0.50, 0.95, 0.99), 1000) AS memory_boundness_index_pct,
            percentile_approx(TFLOPS_per_watt_efficiency, array(0.50, 0.95, 0.99), 1000) AS TFLOPS_per_watt_efficiency_pct,
            percentile_approx(compute_occupancy_index, array(0.50, 0.95, 0.99), 1000) AS compute_occupancy_index_pct,
            percentile_approx(tensor_util_dram_index, array(0.50, 0.95, 0.99), 1000) AS tensor_util_dram_index_pct,
            percentile_approx(tensor_to_gpu_ratio_index, array(0.50, 0.95, 0.99), 1000) AS tensor_to_gpu_ratio_pct,
            percentile_approx(memory_intensity_index, array(0.50, 0.95, 0.99), 1000) AS memory_intensity_pct,
            percentile_approx(power_pct_max_index, array(0.50, 0.95, 0.99), 1000) AS power_pct_max_pct,
            percentile_approx(compute_to_memory_ratio_index, array(0.50, 0.95, 0.99), 1000) AS compute_to_memory_ratio_pct,
            percentile_approx(compute_pct_max_index, array(0.50, 0.95, 0.99), 1000) AS compute_pct_max_pct
        FROM base
        GROUP BY
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name
    ) s
),

final AS (
    SELECT
        r.*,
        nd.node_count_daily_avg,
        u.gpu_util_p50,
        u.gpu_util_p95,
        u.gpu_util_p99,
        u.tensor_util_p50,
        u.tensor_util_p95,
        u.tensor_util_p99,
        u.dram_active_p50,
        u.dram_active_p95,
        u.dram_active_p99,
        u.mem_copy_util_p50,
        u.mem_copy_util_p95,
        u.mem_copy_util_p99,
        u.sm_active_p50,
        u.sm_active_p95,
        u.sm_active_p99,
        u.sm_occupancy_p50,
        u.sm_occupancy_p95,
        u.sm_occupancy_p99,
        d.tensor_tflops_fleet_avg,
        d.tensor_tflops_fleet_p50,
        d.tensor_tflops_fleet_p95,
        d.tensor_tflops_fleet_p99,
        d.tensor_tflops_sm_fleet_p50,
        d.tensor_tflops_sm_fleet_p95,
        d.tensor_tflops_sm_fleet_p99,
        d.chip_power_fleet_p50,
        d.chip_power_fleet_p95,
        d.chip_power_fleet_p99,
        d.redfish_power_fleet_p50,
        d.redfish_power_fleet_p95,
        d.redfish_power_fleet_p99,
        d.vram_usage_fleet_p50,
        d.vram_usage_fleet_p95,
        d.vram_usage_fleet_p99,
        d.tensor_tflops_fleet_p50 / NULLIF(nd.node_count_daily_avg, 0) AS tflops_node_avg_p50,
        d.tensor_tflops_fleet_p95 / NULLIF(nd.node_count_daily_avg, 0) AS tflops_node_avg_p95,
        d.tensor_tflops_fleet_p99 / NULLIF(nd.node_count_daily_avg, 0) AS tflops_node_avg_p99
    FROM final_rowstats r
    LEFT JOIN fleet_daily d
      ON r.day                = d.day
     AND r.region             = d.region
     AND r.region_rollup      = d.region_rollup
     AND r.zone               = d.zone
     AND r.gen                = d.gen
     AND r.is_training        = d.is_training
     AND r.is_multinode       = d.is_multinode
     AND r.product_resolved   = d.product_resolved
     AND r.product_segment    = d.product_segment
     AND r.gpu_count_expected = d.gpu_count_expected
     AND r.customer_segment   = d.customer_segment
     AND r.customer_name      = d.customer_name
    LEFT JOIN node_daily nd
      ON r.day                = nd.day
     AND r.region             = nd.region
     AND r.region_rollup      = nd.region_rollup
     AND r.zone               = nd.zone
     AND r.gen                = nd.gen
     AND r.is_training        = nd.is_training
     AND r.is_multinode       = nd.is_multinode
     AND r.product_resolved   = nd.product_resolved
     AND r.product_segment    = nd.product_segment
     AND r.gpu_count_expected = nd.gpu_count_expected
     AND r.customer_segment   = nd.customer_segment
     AND r.customer_name      = nd.customer_name
    LEFT JOIN util_daily u
      ON r.day                = u.day
     AND r.region             = u.region
     AND r.region_rollup      = u.region_rollup
     AND r.zone               = u.zone
     AND r.gen                = u.gen
     AND r.is_training        = u.is_training
     AND r.is_multinode       = u.is_multinode
     AND r.product_resolved   = u.product_resolved
     AND r.product_segment    = u.product_segment
     AND r.gpu_count_expected = u.gpu_count_expected
     AND r.customer_segment   = u.customer_segment
     AND r.customer_name      = u.customer_name
)
SELECT
    day,
    region,
    region_rollup,
    zone,
    gen,
    is_training,
    is_multinode,
    UPPER(product_resolved) AS product_resolved,
    UPPER(product_segment) AS product_segment,
    gpu_count_expected,
    CONCAT(UPPER(SUBSTRING(customer_segment, 1, 1)), SUBSTRING(customer_segment, 2)) AS customer_segment,
    customer_name,
    peak_power_unit,
    gpu_util_p50,
    gpu_util_p95,
    gpu_util_p99,
    tensor_util_p50,
    tensor_util_p95,
    tensor_util_p99,
    dram_active_p50,
    dram_active_p95,
    dram_active_p99,
    mem_copy_util_p50,
    mem_copy_util_p95,
    mem_copy_util_p99,
    sm_active_p50,
    sm_active_p95,
    sm_active_p99,
    sm_clock_p50,
    sm_clock_p95,
    sm_clock_p99,
    sm_occupancy_p50,
    sm_occupancy_p95,
    sm_occupancy_p99,
    memory_boundness_index_p50,
    memory_boundness_index_p95,
    memory_boundness_index_p99,
    TFLOPS_per_watt_efficiency_p50,
    TFLOPS_per_watt_efficiency_p95,
    TFLOPS_per_watt_efficiency_p99,
    compute_occupancy_index_p50,
    compute_occupancy_index_p95,
    compute_occupancy_index_p99,
    tensor_util_dram_index_p50,
    tensor_util_dram_index_p95,
    tensor_util_dram_index_p99,
    tensor_to_gpu_ratio_p50,
    tensor_to_gpu_ratio_p95,
    tensor_to_gpu_ratio_p99,
    memory_intensity_p50,
    memory_intensity_p95,
    memory_intensity_p99,
    power_pct_max_p50,
    power_pct_max_p95,
    power_pct_max_p99,
    compute_to_memory_ratio_p50,
    compute_to_memory_ratio_p95,
    compute_to_memory_ratio_p99,
    compute_pct_max_p50,
    compute_pct_max_p95,
    compute_pct_max_p99,
    node_count_daily_avg,
    tensor_tflops_fleet_avg,
    tensor_tflops_fleet_p50,
    tensor_tflops_fleet_p95,
    tensor_tflops_fleet_p99,
    tensor_tflops_sm_fleet_p50,
    tensor_tflops_sm_fleet_p95,
    tensor_tflops_sm_fleet_p99,
    chip_power_fleet_p50,
    chip_power_fleet_p95,
    chip_power_fleet_p99,
    redfish_power_fleet_p50,
    redfish_power_fleet_p95,
    redfish_power_fleet_p99,
    vram_usage_fleet_p50,
    vram_usage_fleet_p95,
    vram_usage_fleet_p99,
    tflops_node_avg_p50,
    tflops_node_avg_p95,
    tflops_node_avg_p99
FROM final;
'''


In [17]:
summary_sql_monthly = '''
WITH base AS (
    SELECT
        datestamp,
-- Force date truncation ONCE at the very beginning of the pipeline
        DATE_TRUNC('month', datestamp) AS day,
        region,
        zone,
        trim(customer_segment) AS customer_segment,
        trim(customer_name)    AS customer_name,
        coalesce(trim(cast(is_training  AS string)), 'unknown') AS is_training,
        coalesce(trim(cast(is_multinode AS string)), 'unknown') AS is_multinode,
        trim(product_resolved) AS product_resolved,
        trim(product_segment)  AS product_segment,
        cast(gpu_count_expected AS int) AS gpu_count_expected,
        gen,
        region_rollup,
        tensor_tflops,
        node,
        peak_power_unit,
        gpu_util,
        tensor_util,
        chip_power,
        redfish_power,
        dram_active,
        mem_copy_util,
        vram_usage,
        sm_active,
        sm_clock,
        sm_occupancy,
        memory_boundness_index,
        TFLOPS_per_watt_efficiency,
        compute_occupancy_index,
        tensor_util_dram_index,
        tensor_tflops_sm,
        tensor_to_gpu_ratio_index,
        memory_intensity_index,
        power_pct_max_index,
        compute_to_memory_ratio_index,
        compute_pct_max_index
    FROM {raw_imputed_ref}
),

per_ts AS (
    SELECT
        day,  -- Inherited static value (month)
        datestamp AS ts15,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        SUM(tensor_tflops) AS tensor_tflops_fleet_inst,
        SUM(tensor_tflops_sm) AS tensor_tflops_sm_fleet_inst,
        SUM(chip_power) AS chip_power_fleet_inst,
        SUM(redfish_power) AS redfish_power_fleet_inst,
        SUM(vram_usage) AS vram_usage_fleet_inst,
        COUNT(DISTINCT node) AS node_count
    FROM base
    GROUP BY
        day,
        datestamp,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name
),

fleet_daily AS (
    SELECT
        day,  -- Inherited static value (month)
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        tensor_tflops_fleet_avg,
        element_at(tensor_tflops_fleet_pct, 1) AS tensor_tflops_fleet_p50,
        element_at(tensor_tflops_fleet_pct, 2) AS tensor_tflops_fleet_p95,
        element_at(tensor_tflops_fleet_pct, 3) AS tensor_tflops_fleet_p99,
        element_at(tensor_tflops_sm_fleet_pct, 1) AS tensor_tflops_sm_fleet_p50,
        element_at(tensor_tflops_sm_fleet_pct, 2) AS tensor_tflops_sm_fleet_p95,
        element_at(tensor_tflops_sm_fleet_pct, 3) AS tensor_tflops_sm_fleet_p99,
        element_at(chip_power_fleet_pct, 1) AS chip_power_fleet_p50,
        element_at(chip_power_fleet_pct, 2) AS chip_power_fleet_p95,
        element_at(chip_power_fleet_pct, 3) AS chip_power_fleet_p99,
        element_at(redfish_power_fleet_pct, 1) AS redfish_power_fleet_p50,
        element_at(redfish_power_fleet_pct, 2) AS redfish_power_fleet_p95,
        element_at(redfish_power_fleet_pct, 3) AS redfish_power_fleet_p99,
        element_at(vram_usage_fleet_pct, 1) AS vram_usage_fleet_p50,
        element_at(vram_usage_fleet_pct, 2) AS vram_usage_fleet_p95,
        element_at(vram_usage_fleet_pct, 3) AS vram_usage_fleet_p99
    FROM (
        SELECT
            day,  -- Inherited static value (month)
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name,
            SUM(tensor_tflops_fleet_inst) / 96.0 AS tensor_tflops_fleet_avg,
            percentile_approx(tensor_tflops_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS tensor_tflops_fleet_pct,
            percentile_approx(tensor_tflops_sm_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS tensor_tflops_sm_fleet_pct,
            percentile_approx(chip_power_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS chip_power_fleet_pct,
            percentile_approx(redfish_power_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS redfish_power_fleet_pct,
            percentile_approx(vram_usage_fleet_inst, array(0.50, 0.95, 0.99), 1000) AS vram_usage_fleet_pct
        FROM per_ts
        GROUP BY
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name
    ) t
),

node_daily AS (
    SELECT
        day,  -- Inherited static value (month)
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        AVG(node_count) AS node_count_daily_avg
    FROM per_ts
    GROUP BY
        day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name
),

-- =====================================================================
-- METHOD 2 (bounded utilization ratios only): average each node to its
-- CALENDAR-DAY mean first (the reclaim grain), THEN percentile across all
-- node-day means WITHIN THE MONTH. Note: `day` here is the month bucket,
-- so the per-node daily mean is keyed on cal_day = DATE_TRUNC('day', ts).
-- This is the statistically valid monthly percentile (recomputed from the
-- node-day base), NOT a stack of daily percentiles.
-- Applies to: gpu_util, tensor_util, dram_active, mem_copy_util,
--             sm_active, sm_occupancy.
-- (power_pct_max and compute_pct_max intentionally stay on raw-grain
--  percentiles -- peak is the physical constraint.)
-- =====================================================================
node_util_daily AS (
    SELECT
        day,  -- month bucket
        DATE_TRUNC('day', datestamp) AS cal_day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        node,
        AVG(LEAST(gpu_util, 1))      AS gpu_util_node,
        AVG(LEAST(tensor_util, 1))   AS tensor_util_node,
        AVG(LEAST(dram_active, 1))   AS dram_active_node,
        AVG(LEAST(mem_copy_util, 1)) AS mem_copy_util_node,
        AVG(LEAST(sm_active, 1))     AS sm_active_node,
        AVG(LEAST(sm_occupancy, 1))  AS sm_occupancy_node
    FROM base
    GROUP BY
        day,
        DATE_TRUNC('day', datestamp),
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        node
),

util_daily AS (
    SELECT
        day,  -- month bucket
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        element_at(gpu_util_pct, 1)      AS gpu_util_p50,
        element_at(gpu_util_pct, 2)      AS gpu_util_p95,
        element_at(gpu_util_pct, 3)      AS gpu_util_p99,
        element_at(tensor_util_pct, 1)   AS tensor_util_p50,
        element_at(tensor_util_pct, 2)   AS tensor_util_p95,
        element_at(tensor_util_pct, 3)   AS tensor_util_p99,
        element_at(dram_active_pct, 1)   AS dram_active_p50,
        element_at(dram_active_pct, 2)   AS dram_active_p95,
        element_at(dram_active_pct, 3)   AS dram_active_p99,
        element_at(mem_copy_util_pct, 1) AS mem_copy_util_p50,
        element_at(mem_copy_util_pct, 2) AS mem_copy_util_p95,
        element_at(mem_copy_util_pct, 3) AS mem_copy_util_p99,
        element_at(sm_active_pct, 1)     AS sm_active_p50,
        element_at(sm_active_pct, 2)     AS sm_active_p95,
        element_at(sm_active_pct, 3)     AS sm_active_p99,
        element_at(sm_occupancy_pct, 1)  AS sm_occupancy_p50,
        element_at(sm_occupancy_pct, 2)  AS sm_occupancy_p95,
        element_at(sm_occupancy_pct, 3)  AS sm_occupancy_p99
    FROM (
        SELECT
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name,
            percentile_approx(gpu_util_node,      array(0.50, 0.95, 0.99), 1000) AS gpu_util_pct,
            percentile_approx(tensor_util_node,   array(0.50, 0.95, 0.99), 1000) AS tensor_util_pct,
            percentile_approx(dram_active_node,   array(0.50, 0.95, 0.99), 1000) AS dram_active_pct,
            percentile_approx(mem_copy_util_node, array(0.50, 0.95, 0.99), 1000) AS mem_copy_util_pct,
            percentile_approx(sm_active_node,     array(0.50, 0.95, 0.99), 1000) AS sm_active_pct,
            percentile_approx(sm_occupancy_node,  array(0.50, 0.95, 0.99), 1000) AS sm_occupancy_pct
        FROM node_util_daily
        GROUP BY
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name
    ) u
),

final_rowstats AS (
    SELECT
        day,
        region,
        region_rollup,
        zone,
        gen,
        is_training,
        is_multinode,
        product_resolved,
        product_segment,
        gpu_count_expected,
        customer_segment,
        customer_name,
        peak_power_unit,
        element_at(sm_clock_pct, 1) AS sm_clock_p50,
        element_at(sm_clock_pct, 2) AS sm_clock_p95,
        element_at(sm_clock_pct, 3) AS sm_clock_p99,
        element_at(memory_boundness_index_pct, 1) AS memory_boundness_index_p50,
        element_at(memory_boundness_index_pct, 2) AS memory_boundness_index_p95,
        element_at(memory_boundness_index_pct, 3) AS memory_boundness_index_p99,
        element_at(TFLOPS_per_watt_efficiency_pct, 1) AS TFLOPS_per_watt_efficiency_p50,
        element_at(TFLOPS_per_watt_efficiency_pct, 2) AS TFLOPS_per_watt_efficiency_p95,
        element_at(TFLOPS_per_watt_efficiency_pct, 3) AS TFLOPS_per_watt_efficiency_p99,
        element_at(compute_occupancy_index_pct, 1) AS compute_occupancy_index_p50,
        element_at(compute_occupancy_index_pct, 2) AS compute_occupancy_index_p95,
        element_at(compute_occupancy_index_pct, 3) AS compute_occupancy_index_p99,
        element_at(tensor_util_dram_index_pct, 1) AS tensor_util_dram_index_p50,
        element_at(tensor_util_dram_index_pct, 2) AS tensor_util_dram_index_p95,
        element_at(tensor_util_dram_index_pct, 3) AS tensor_util_dram_index_p99,
        element_at(tensor_to_gpu_ratio_pct, 1) AS tensor_to_gpu_ratio_p50,
        element_at(tensor_to_gpu_ratio_pct, 2) AS tensor_to_gpu_ratio_p95,
        element_at(tensor_to_gpu_ratio_pct, 3) AS tensor_to_gpu_ratio_p99,
        element_at(memory_intensity_pct, 1) AS memory_intensity_p50,
        element_at(memory_intensity_pct, 2) AS memory_intensity_p95,
        element_at(memory_intensity_pct, 3) AS memory_intensity_p99,
        element_at(power_pct_max_pct, 1) AS power_pct_max_p50,
        element_at(power_pct_max_pct, 2) AS power_pct_max_p95,
        element_at(power_pct_max_pct, 3) AS power_pct_max_p99,
        element_at(compute_to_memory_ratio_pct, 1) AS compute_to_memory_ratio_p50,
        element_at(compute_to_memory_ratio_pct, 2) AS compute_to_memory_ratio_p95,
        element_at(compute_to_memory_ratio_pct, 3) AS compute_to_memory_ratio_p99,
        element_at(compute_pct_max_pct, 1) AS compute_pct_max_p50,
        element_at(compute_pct_max_pct, 2) AS compute_pct_max_p95,
        element_at(compute_pct_max_pct, 3) AS compute_pct_max_p99
    FROM (
        SELECT
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name,
            AVG(peak_power_unit) AS peak_power_unit,
            percentile_approx(sm_clock, array(0.50, 0.95, 0.99), 1000) AS sm_clock_pct,
            percentile_approx(memory_boundness_index, array(0.50, 0.95, 0.99), 1000) AS memory_boundness_index_pct,
            percentile_approx(TFLOPS_per_watt_efficiency, array(0.50, 0.95, 0.99), 1000) AS TFLOPS_per_watt_efficiency_pct,
            percentile_approx(compute_occupancy_index, array(0.50, 0.95, 0.99), 1000) AS compute_occupancy_index_pct,
            percentile_approx(tensor_util_dram_index, array(0.50, 0.95, 0.99), 1000) AS tensor_util_dram_index_pct,
            percentile_approx(tensor_to_gpu_ratio_index, array(0.50, 0.95, 0.99), 1000) AS tensor_to_gpu_ratio_pct,
            percentile_approx(memory_intensity_index, array(0.50, 0.95, 0.99), 1000) AS memory_intensity_pct,
            percentile_approx(power_pct_max_index, array(0.50, 0.95, 0.99), 1000) AS power_pct_max_pct,
            percentile_approx(compute_to_memory_ratio_index, array(0.50, 0.95, 0.99), 1000) AS compute_to_memory_ratio_pct,
            percentile_approx(compute_pct_max_index, array(0.50, 0.95, 0.99), 1000) AS compute_pct_max_pct
        FROM base
        GROUP BY
            day,
            region,
            region_rollup,
            zone,
            gen,
            is_training,
            is_multinode,
            product_resolved,
            product_segment,
            gpu_count_expected,
            customer_segment,
            customer_name
    ) s
),

final AS (
    SELECT
        r.*,
        nd.node_count_daily_avg,
        u.gpu_util_p50,
        u.gpu_util_p95,
        u.gpu_util_p99,
        u.tensor_util_p50,
        u.tensor_util_p95,
        u.tensor_util_p99,
        u.dram_active_p50,
        u.dram_active_p95,
        u.dram_active_p99,
        u.mem_copy_util_p50,
        u.mem_copy_util_p95,
        u.mem_copy_util_p99,
        u.sm_active_p50,
        u.sm_active_p95,
        u.sm_active_p99,
        u.sm_occupancy_p50,
        u.sm_occupancy_p95,
        u.sm_occupancy_p99,
        d.tensor_tflops_fleet_avg,
        d.tensor_tflops_fleet_p50,
        d.tensor_tflops_fleet_p95,
        d.tensor_tflops_fleet_p99,
        d.tensor_tflops_sm_fleet_p50,
        d.tensor_tflops_sm_fleet_p95,
        d.tensor_tflops_sm_fleet_p99,
        d.chip_power_fleet_p50,
        d.chip_power_fleet_p95,
        d.chip_power_fleet_p99,
        d.redfish_power_fleet_p50,
        d.redfish_power_fleet_p95,
        d.redfish_power_fleet_p99,
        d.vram_usage_fleet_p50,
        d.vram_usage_fleet_p95,
        d.vram_usage_fleet_p99,
        d.tensor_tflops_fleet_p50 / NULLIF(nd.node_count_daily_avg, 0) AS tflops_node_avg_p50,
        d.tensor_tflops_fleet_p95 / NULLIF(nd.node_count_daily_avg, 0) AS tflops_node_avg_p95,
        d.tensor_tflops_fleet_p99 / NULLIF(nd.node_count_daily_avg, 0) AS tflops_node_avg_p99
    FROM final_rowstats r
    LEFT JOIN fleet_daily d
      ON r.day                = d.day
     AND r.region             = d.region
     AND r.region_rollup      = d.region_rollup
     AND r.zone               = d.zone
     AND r.gen                = d.gen
     AND r.is_training        = d.is_training
     AND r.is_multinode       = d.is_multinode
     AND r.product_resolved   = d.product_resolved
     AND r.product_segment    = d.product_segment
     AND r.gpu_count_expected = d.gpu_count_expected
     AND r.customer_segment   = d.customer_segment
     AND r.customer_name      = d.customer_name
    LEFT JOIN node_daily nd
      ON r.day                = nd.day
     AND r.region             = nd.region
     AND r.region_rollup      = nd.region_rollup
     AND r.zone               = nd.zone
     AND r.gen                = nd.gen
     AND r.is_training        = nd.is_training
     AND r.is_multinode       = nd.is_multinode
     AND r.product_resolved   = nd.product_resolved
     AND r.product_segment    = nd.product_segment
     AND r.gpu_count_expected = nd.gpu_count_expected
     AND r.customer_segment   = nd.customer_segment
     AND r.customer_name      = nd.customer_name
    LEFT JOIN util_daily u
      ON r.day                = u.day
     AND r.region             = u.region
     AND r.region_rollup      = u.region_rollup
     AND r.zone               = u.zone
     AND r.gen                = u.gen
     AND r.is_training        = u.is_training
     AND r.is_multinode       = u.is_multinode
     AND r.product_resolved   = u.product_resolved
     AND r.product_segment    = u.product_segment
     AND r.gpu_count_expected = u.gpu_count_expected
     AND r.customer_segment   = u.customer_segment
     AND r.customer_name      = u.customer_name
)
SELECT
    day,
    region,
    region_rollup,
    zone,
    gen,
    is_training,
    is_multinode,
    UPPER(product_resolved) AS product_resolved,
    UPPER(product_segment) AS product_segment,
    gpu_count_expected,
    CONCAT(UPPER(SUBSTRING(customer_segment, 1, 1)), SUBSTRING(customer_segment, 2)) AS customer_segment,
    customer_name,
    peak_power_unit,
    gpu_util_p50,
    gpu_util_p95,
    gpu_util_p99,
    tensor_util_p50,
    tensor_util_p95,
    tensor_util_p99,
    dram_active_p50,
    dram_active_p95,
    dram_active_p99,
    mem_copy_util_p50,
    mem_copy_util_p95,
    mem_copy_util_p99,
    sm_active_p50,
    sm_active_p95,
    sm_active_p99,
    sm_clock_p50,
    sm_clock_p95,
    sm_clock_p99,
    sm_occupancy_p50,
    sm_occupancy_p95,
    sm_occupancy_p99,
    memory_boundness_index_p50,
    memory_boundness_index_p95,
    memory_boundness_index_p99,
    TFLOPS_per_watt_efficiency_p50,
    TFLOPS_per_watt_efficiency_p95,
    TFLOPS_per_watt_efficiency_p99,
    compute_occupancy_index_p50,
    compute_occupancy_index_p95,
    compute_occupancy_index_p99,
    tensor_util_dram_index_p50,
    tensor_util_dram_index_p95,
    tensor_util_dram_index_p99,
    tensor_to_gpu_ratio_p50,
    tensor_to_gpu_ratio_p95,
    tensor_to_gpu_ratio_p99,
    memory_intensity_p50,
    memory_intensity_p95,
    memory_intensity_p99,
    power_pct_max_p50,
    power_pct_max_p95,
    power_pct_max_p99,
    compute_to_memory_ratio_p50,
    compute_to_memory_ratio_p95,
    compute_to_memory_ratio_p99,
    compute_pct_max_p50,
    compute_pct_max_p95,
    compute_pct_max_p99,
    node_count_daily_avg,
    tensor_tflops_fleet_avg,
    tensor_tflops_fleet_p50,
    tensor_tflops_fleet_p95,
    tensor_tflops_fleet_p99,
    tensor_tflops_sm_fleet_p50,
    tensor_tflops_sm_fleet_p95,
    tensor_tflops_sm_fleet_p99,
    chip_power_fleet_p50,
    chip_power_fleet_p95,
    chip_power_fleet_p99,
    redfish_power_fleet_p50,
    redfish_power_fleet_p95,
    redfish_power_fleet_p99,
    vram_usage_fleet_p50,
    vram_usage_fleet_p95,
    vram_usage_fleet_p99,
    tflops_node_avg_p50,
    tflops_node_avg_p95,
    tflops_node_avg_p99
FROM final;
'''


In [18]:
#spark.catalog.clearCache()

In [ ]:
## WRITE TO NESSIE & STARROCKS
# ========================================
# 1. Read from sandbox source table
# 2. Build daily summary from persisted raw table  
# 3. Write daily summary
# 4. Build monthly summary from same persisted source
# 5. Write monthly summary
# 6. Release caches and print schemas

import time
from pyspark import StorageLevel
from pyspark.sql import functions as F

def timed_count(df, label):
    t0 = time.time()
    n = df.count()
    dt = time.time() - t0
    print(f"✓ {label}: {n:,} rows materialized in {dt:.1f}s ({dt/60:.1f} min)")
    return n

def safe_unpersist(name):
    if name in globals():
        try:
            obj = globals()[name]
            if hasattr(obj, "is_cached") and obj.is_cached:
                obj.unpersist(blocking=True)
                print(f"✓ Unpersisted {name}")
        except Exception as e:
            print(f"Note: could not unpersist {name}: {e}")

def overwrite_iceberg_table(df, target_dataset, label, repartition_n=None):
    print("=" * 60)
    print(f"WRITING {label} TO NESSIE")
    print("=" * 60)
    print(f"Target dataset: {target_dataset}")

    t0 = time.time()
    ness.db_prepare(target_dataset)
    table_ref = ness.table_ref(target_dataset)
    print(f"✓ Dataset prepared: {table_ref}")

    df_to_write = df
    if repartition_n:
        print(f"Repartitioning to {repartition_n} partitions before write...")
        df_to_write = df_to_write.repartition(repartition_n)

    print("Materializing write DataFrame...")
    df_to_write = df_to_write.persist(StorageLevel.DISK_ONLY)
    timed_count(df_to_write, f"{label} write DataFrame")

    print("Dropping target table if it exists...")
    spark.sql(f"DROP TABLE IF EXISTS {table_ref}")

    print("Writing via DataFrame writer...")
    (
        df_to_write.write
        .format("iceberg")
        .mode("overwrite")
        .saveAsTable(table_ref)
    )

    df_to_write.unpersist(blocking=True)

    dt = time.time() - t0
    print(f"✓ {label} write completed in {dt:.1f}s ({dt/60:.1f} min)")

    print("Verifying...")
    spark.table(table_ref).limit(1).count()
    print("✓ Verification successful")

    return table_ref, dt

# ========================================
# CLEAN UP STALE CACHES FROM INTERRUPTED RUNS
# ========================================
for _name in [
    "df_imputed_summary",
    "df_imputed_summary_monthly",
    "df_src",
    "sdf_num",
    "train_sdf",
    "test_sdf",
    "assembled_train",
    "assembled_test",
    "train_sdf_final",
    "test_sdf_final",
    "assembled_train_final",
    "assembled_test_final",
]:
    safe_unpersist(_name)

spark.catalog.clearCache()
print("✓ Cleared Spark SQL cache")

# ========================================
# PART 1: READ SOURCE DATA FROM SANDBOX TABLE
# ========================================
raw_source_dataset = "sandbox.sandbox_finance.dcgm_metrics_raw_imputed_v2"
print("" + "=" * 60)
print("READING SOURCE DATA")
print("=" * 60)
print(f"Source dataset: {raw_source_dataset}")

# Read the source data and cache it
df_src = spark.table(raw_source_dataset).persist(StorageLevel.MEMORY_AND_DISK)
src_count = timed_count(df_src, "Source data from sandbox table")

# Create temporary view for SQL queries
df_src.createOrReplaceTempView("dcgm_imputed_src")
print("✓ Created temp view dcgm_imputed_src")

print("🔍 DEBUG: Column names from source table:")
for i, col_name in enumerate(df_src.columns):
    if "index" in col_name or "pct" in col_name or "%" in col_name:
        print(f"  {i}: {col_name}")
print(f"Total columns: {len(df_src.columns)}")

# ========================================
# PART 2: CREATE DAILY SUMMARY
# ========================================
print("" + "=" * 60)
print("PART 2: CREATING DAILY SUMMARY DATAFRAME")
print("=" * 60)

# Ensure summary_sql is defined - if not, provide a fallback
if "summary_sql" not in globals():
    print("⚠️ WARNING: summary_sql not found in globals. Using default query.")
    summary_sql = """SELECT 
    DATE_TRUNC('day', datestamp) AS day,
    region,
    zone,
    customer_segment,
    customer_name,
    is_training,
    is_multinode,
    product_resolved,
    product_segment,
    gpu_count_expected,
    COUNT(*) as record_count,
    AVG(gpu_util) as avg_gpu_util,
    AVG(tensor_util) as avg_tensor_util,
    AVG(redfish_power) as avg_redfish_power,
    AVG(chip_power) as avg_chip_power,
    AVG(tensor_tflops) as avg_tensor_tflops
FROM {raw_imputed_ref}
GROUP BY 1,2,3,4,5,6,7,8,9,10"""

daily_start = time.time()
resolved_summary_sql = summary_sql.format(raw_imputed_ref="dcgm_imputed_src")
df_imputed_summary = spark.sql(resolved_summary_sql).persist(StorageLevel.MEMORY_AND_DISK)
timed_count(df_imputed_summary, "Daily summary")
daily_time = time.time() - daily_start
print(f"✓ Daily summary created in {daily_time:.1f}s ({daily_time/60:.1f} min)")

# ========================================
# PART 3: WRITE DAILY SUMMARY
# ========================================
summary_target_dataset = "sandbox_finance.dcgm_metrics_summary_imputed_daily_v2"
summary_ref, daily_write_time = overwrite_iceberg_table(
    df=df_imputed_summary,
    target_dataset=summary_target_dataset,
    label="daily summary",
    repartition_n=200,  # optional tuning knob
)

# ========================================
# PART 4: CREATE MONTHLY SUMMARY
# ========================================
print("" + "=" * 60)
print("PART 4: CREATING MONTHLY SUMMARY DATAFRAME")
print("=" * 60)

# Ensure summary_sql_monthly is defined - if not, provide a fallback
if "summary_sql_monthly" not in globals():
    print("⚠️ WARNING: summary_sql_monthly not found in globals. Using default query.")
    summary_sql_monthly = """SELECT 
    DATE_TRUNC('month', datestamp) AS day,
    region,
    zone,
    customer_segment,
    customer_name,
    is_training,
    is_multinode,
    product_resolved,
    product_segment,
    gpu_count_expected,
    COUNT(*) as record_count,
    AVG(gpu_util) as avg_gpu_util,
    AVG(tensor_util) as avg_tensor_util,
    AVG(redfish_power) as avg_redfish_power,
    AVG(chip_power) as avg_chip_power,
    AVG(tensor_tflops) as avg_tensor_tflops
FROM {raw_imputed_ref}
GROUP BY 1,2,3,4,5,6,7,8,9,10"""

monthly_start = time.time()
resolved_summary_sql_monthly = summary_sql_monthly.format(raw_imputed_ref="dcgm_imputed_src")
df_imputed_summary_monthly = spark.sql(resolved_summary_sql_monthly).persist(StorageLevel.MEMORY_AND_DISK)
timed_count(df_imputed_summary_monthly, "Monthly summary")
monthly_time = time.time() - monthly_start
print(f"✓ Monthly summary created in {monthly_time:.1f}s ({monthly_time/60:.1f} min)")

# ========================================
# PART 5: WRITE MONTHLY SUMMARY
# ========================================
monthly_target_dataset = "sandbox_finance.dcgm_metrics_summary_imputed_monthly_v2"
monthly_ref, monthly_write_time = overwrite_iceberg_table(
    df=df_imputed_summary_monthly,
    target_dataset=monthly_target_dataset,
    label="monthly summary",
    repartition_n=100,  # optional tuning knob
)

# ========================================
# CLEANUP
# ========================================
df_imputed_summary.unpersist(blocking=True)
df_imputed_summary_monthly.unpersist(blocking=True)
df_src.unpersist(blocking=True)
print("✓ Released summary/source caches")

# ========================================
# PRINT SCHEMAS
# ========================================
print("" + "=" * 60)
print("SCHEMA INFORMATION")
print("=" * 60)

print(f"Schema for {raw_source_dataset}:")
print("-" * 60)
spark.table(raw_source_dataset).printSchema()

print(f"Schema for {summary_ref}:")
print("-" * 60)
spark.table(summary_ref).printSchema()

print(f"Schema for {monthly_ref}:")
print("-" * 60)
spark.table(monthly_ref).printSchema()

print("=" * 60)
print("✅ ALL OPERATIONS COMPLETED SUCCESSFULLY")
print("=" * 60)

# ========================================
# SUMMARY REPORT
# ========================================
print("📊 EXECUTION SUMMARY:")
print(f"  • Source records read: {src_count:,}")
print(f"  • Daily summary created: {daily_time:.1f}s")
print(f"  • Daily summary written: {daily_write_time:.1f}s")  
print(f"  • Monthly summary created: {monthly_time:.1f}s")
print(f"  • Monthly summary written: {monthly_write_time:.1f}s")
print(f"  • Total execution time: {(daily_time + daily_write_time + monthly_time + monthly_write_time):.1f}s")
print("=" * 60)

SyntaxError: unterminated string literal (detected at line 99) (2956985581.py, line 99)

In [ ]:
# ========================================
# FINAL CLEANUP
# ========================================

print("="*60)
print("FINAL CLEANUP")
print("="*60)

for df_name in ["df_raw_sample", "df_summary_clean", "_df_binned", "_df_binned_time"]:
    if df_name in globals():
        try:
            globals()[df_name].unpersist()
            print(f"✓ Unpersisted {df_name}")
        except Exception as e:
            print(f"Note: could not unpersist {df_name}: {e}")

try:
    spark.catalog.clearCache()
    print("✓ Cleared Spark cache")
except Exception as e:
    print(f"Note: could not clear Spark cache: {e}")

print("="*60)
print("✓ CLEANUP COMPLETE")
print("="*60)


In [ ]:
# # CLASSIFICATION ANALYSIS: training_inference
# # ========================================
# # Analyze feature characteristics that separate training_inference categories
# # Uses full df_imputed dataset with Spark MLlib + scikit-learn hybrid approach

# # Install packages (SparkCaster kernel workaround)
# import subprocess
# import sys
# import importlib
# import site

# def install_and_import(package_name, import_name=None):
#     """Install package and reload site-packages to make it importable"""
#     if import_name is None:
#         import_name = package_name
    
#     try:
#         return __import__(import_name)
#     except ImportError:
#         print(f"Installing {package_name}...")
#         subprocess.check_call([
#             sys.executable, '-m', 'pip', 'install', 
#             '--user', '--upgrade', package_name
#         ])
#         # Reload site packages to pick up newly installed packages
#         importlib.reload(site)
#         # Force reimport of sys.path
#         import site as site_module
#         site_module.main()
#         return __import__(import_name)

# # Install required packages
# print("Installing required packages...")
# install_and_import('matplotlib')
# install_and_import('seaborn')
# install_and_import('plotly')
# install_and_import('scikit-learn', 'sklearn')
# print("✓ Packages installed")

# import time
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from pyspark.ml import Pipeline
# from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler as MLStandardScaler
# from pyspark.ml.classification import RandomForestClassifier, LogisticRegression as MLLogisticRegression, GBTClassifier
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# from pyspark.sql import functions as F
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import classification_report, confusion_matrix
# import plotly.express as px
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# print("="*60)
# print("CLASSIFICATION ANALYSIS: training_inference")
# print("="*60)

# # ========================================
# # Configuration
# # ========================================
# TARGET_COL = "training_inference"
# SEED = 42

# # Final stratified sample size (settled on 0.1% after learning curve analysis)
# FINAL_SAMPLE_FRAC = 0.001  # 0.1%
# SAMPLE_FOR_VIZ = 0.01  # 1% for visualization (will be further reduced if needed)

# print(f"Target variable: {TARGET_COL}")
# print(f"Using full imputed dataset: {imputed_row_count:,} rows")
# print(f"Final sample fraction: {FINAL_SAMPLE_FRAC*100:.1f}%")

# # ========================================
# # [1/6] Data Preparation
# # ========================================
# print("[1/6] Preparing data...")

# # Check class distribution (expensive on full data; keep for visibility)
# print(f"Class distribution in {TARGET_COL}:")
# class_dist = df_imputed.groupBy(TARGET_COL).count().orderBy("count", ascending=False)
# class_dist.show()

# # Select features - numeric and categorical
# numeric_features = [
#     "gpu_util", "tensor_util", "chip_power", "redfish_power",
#     "tensor_tflops", "vram_usage", "mem_copy_util", "sm_active",
#     "sm_occupancy", "dram_active", "sm_clock", "tensor_tflops_sm",
#     "gpu_peak_power_node_watts", "peak_tflops_unit", "peak_power_unit",
#     # Engineered features
#     "tensor_to_gpu_ratio_index", "memory_intensity_index", "compute_to_memory_ratio_index",
#     "power_pct_max_index", "compute_pct_max_index"
# ]

# categorical_features = [
#     "product_segment"
# ]

# print(f"Numeric features: {len(numeric_features)}")
# print(f"Categorical features: {len(categorical_features)}")

# # Filter to only rows with non-null target
# df_classification = df_imputed.filter(F.col(TARGET_COL).isNotNull())

# # Handle nulls in features (fill with 0 for numeric, "unknown" for categorical)
# for col in numeric_features:
#     if col in df_classification.columns:
#         df_classification = df_classification.fillna({col: 0})

# for col in categorical_features:
#     if col in df_classification.columns:
#         df_classification = df_classification.fillna({col: "unknown"})

# print(f"✓ Data prepared")

# # ========================================
# # [2/6] Stratified Sampling Helper
# # ========================================
# print("[2/6] Preparing stratified sampling...")

# # Build fractions dict for stratified sampling (collect class labels only)
# classes = [row[TARGET_COL] for row in df_classification.select(TARGET_COL).distinct().collect()]
# print(f"Classes: {classes}")

# def stratified_sample(df, target_col, frac, seed=SEED):
#     """Stratified sample by target_col using sampleBy with uniform fraction."""
#     fractions = {c: frac for c in classes}
#     return df.sampleBy(target_col, fractions=fractions, seed=seed)

# # ========================================
# # [3/6] Feature Encoding Pipeline (fit once)
# # ========================================
# print("[3/6] Fitting encoding pipeline (on full data)...")

# # String index the target variable
# # NOTE: Fit on full data to keep label mapping stable across samples

# target_indexer = StringIndexer(inputCol=TARGET_COL, outputCol="label")

# # Index categorical features
# indexers = []
# indexed_cols = []
# for col in categorical_features:
#     if col in df_classification.columns:
#         indexer = StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
#         indexers.append(indexer)
#         indexed_cols.append(f"{col}_indexed")

# # OneHot encode categorical features
# encoders = []
# encoded_cols = []
# for col in indexed_cols:
#     encoder = OneHotEncoder(inputCol=col, outputCol=f"{col}_encoded")
#     encoders.append(encoder)
#     encoded_cols.append(f"{col}_encoded")

# # Build encoding pipeline
# encoding_stages = [target_indexer] + indexers + encoders
# encoding_pipeline = Pipeline(stages=encoding_stages)
# encoding_model = encoding_pipeline.fit(df_classification)

# print(f"✓ Encoded {len(categorical_features)} categorical features")

# # Combine numeric and encoded categorical features
# all_feature_cols = [col for col in numeric_features if col in df_classification.columns] + encoded_cols

# assembler = VectorAssembler(
#     inputCols=all_feature_cols,
#     outputCol="features_raw",
#     handleInvalid="keep"
# )

# # Scaling for LR/GBT (trees don't require scaling but OK to keep consistent)
# scaler = MLStandardScaler(
#     inputCol="features_raw",
#     outputCol="features",
#     withStd=True,
#     withMean=False
# )

# # # ========================================
# # # [COMMENTED OUT] Learning Curve (GBT) - Already completed, settled on 0.1%
# # # ========================================
# # # print("[4/9] Learning curve with GBTClassifier...")
# # 
# # # LC_START = 0.001  # 0.1%
# # # LC_END = 0.05     # 5%
# # # LC_STEP = 0.005   # 0.5%
# # 
# # # lc_fracs = []
# # # frac = LC_START
# # # while frac <= LC_END + 1e-9:
# # #     lc_fracs.append(round(frac, 6))
# # #     frac += LC_STEP
# # 
# # # lc_results = []
# # 
# # # # Evaluators
# # # lc_evaluator_acc = MulticlassClassificationEvaluator(
# # #     labelCol="label", predictionCol="prediction", metricName="accuracy"
# # # )
# # # lc_evaluator_f1 = MulticlassClassificationEvaluator(
# # #     labelCol="label", predictionCol="prediction", metricName="f1"
# # # )
# # 
# # # for frac in lc_fracs:
# # #     print(f"→ Sample size: {frac*100:.2f}%")
# # #     sample_df = stratified_sample(df_classification, TARGET_COL, frac, seed=SEED)
# # 
# # #     # Encode + assemble + scale on sample
# # #     df_encoded = encoding_model.transform(sample_df)
# # #     df_features = assembler.transform(df_encoded)
# # #     scaler_model = scaler.fit(df_features)
# # #     df_scaled = scaler_model.transform(df_features)
# # 
# # #     df_ml = df_scaled.select("features", "label", TARGET_COL)
# # 
# # #     # Split train/test
# # #     train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=SEED)
# # 
# # #     # GBT classifier
# # #     gbt = GBTClassifier(
# # #         featuresCol="features",
# # #         labelCol="label",
# # #         maxIter=50,
# # #         maxDepth=5,
# # #         stepSize=0.1,
# # #         subsamplingRate=0.8,
# # #         seed=SEED
# # #     )
# # 
# # #     start = time.time()
# # #     gbt_model = gbt.fit(train_df)
# # #     train_time = time.time() - start
# # 
# # #     preds = gbt_model.transform(test_df)
# # #     acc = lc_evaluator_acc.evaluate(preds)
# # #     f1 = lc_evaluator_f1.evaluate(preds)
# # 
# # #     lc_results.append((frac, acc, f1, train_time))
# # #     print(f"    Accuracy: {acc:.4f} | F1: {f1:.4f} | Time: {train_time:.1f}s")
# # 
# # # lc_results_df = pd.DataFrame(lc_results, columns=["sample_frac", "accuracy", "f1", "train_time_s"])
# # # print("Learning curve results:")
# # # print(lc_results_df.to_string(index=False))

# # ========================================
# # [4/6] Final Stratified Sample for Full Training
# # ========================================
# print("[4/6] Applying final stratified sample...")
# print(f"Final sample fraction: {FINAL_SAMPLE_FRAC*100:.2f}%")

# df_sampled = stratified_sample(df_classification, TARGET_COL, FINAL_SAMPLE_FRAC, seed=SEED)

# # ========================================
# # [5/6] Feature Engineering & Assembly
# # ========================================
# print("[5/6] Encoding and assembling features...")

# # Apply encoding model
# _df_encoded = encoding_model.transform(df_sampled)

# # Assemble and scale features
# df_features = assembler.transform(_df_encoded)
# scaler_model_final = scaler.fit(df_features)
# df_scaled = scaler_model_final.transform(df_features)

# # Select only needed columns
# df_ml = df_scaled.select("features", "label", TARGET_COL)
# df_ml.cache()
# _ = df_ml.count()

# print(f"✓ Features assembled and scaled")

# # Split train/test
# train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=SEED)
# train_df.cache()
# test_df.cache()

# train_count = train_df.count()
# test_count = test_df.count()

# print(f"Train: {train_count:,} | Test: {test_count:,}")

# # ========================================
# # [6/6] Model Training with Spark MLlib
# # ========================================
# print("[6/6] Training classification models...")

# evaluator_acc = MulticlassClassificationEvaluator(
#     labelCol="label", predictionCol="prediction", metricName="accuracy"
# )
# evaluator_f1 = MulticlassClassificationEvaluator(
#     labelCol="label", predictionCol="prediction", metricName="f1"
# )

# results = []

# # Model 1: Logistic Regression (Fast baseline - trains first)
# print("  [1/3] Training Logistic Regression...")
# lr_start = time.time()
# lr = MLLogisticRegression(
#     featuresCol="features",
#     labelCol="label",
#     maxIter=100,
#     regParam=0.01
# )
# lr_model = lr.fit(train_df)
# lr_time = time.time() - lr_start

# lr_pred = lr_model.transform(test_df)
# lr_acc = evaluator_acc.evaluate(lr_pred)
# lr_f1 = evaluator_f1.evaluate(lr_pred)

# print(f"    Accuracy: {lr_acc:.4f} | F1: {lr_f1:.4f} | Time: {lr_time:.1f}s")
# results.append(("Logistic Regression", lr_acc, lr_f1, lr_time))

# # Model 2: Random Forest (For feature importance - trains second)
# print("  [2/3] Training Random Forest...")
# rf_start = time.time()
# rf = RandomForestClassifier(
#     featuresCol="features",
#     labelCol="label",
#     numTrees=100,
#     maxDepth=10,
#     seed=SEED
# )
# rf_model = rf.fit(train_df)
# rf_time = time.time() - rf_start

# rf_pred = rf_model.transform(test_df)
# rf_acc = evaluator_acc.evaluate(rf_pred)
# rf_f1 = evaluator_f1.evaluate(rf_pred)

# print(f"    Accuracy: {rf_acc:.4f} | F1: {rf_f1:.4f} | Time: {rf_time:.1f}s")
# results.append(("Random Forest", rf_acc, rf_f1, rf_time))

# # Model 3: GBT Classifier
# print("  [3/3] Training GBT Classifier...")
# gbt_start = time.time()
# gbt = GBTClassifier(
#     featuresCol="features",
#     labelCol="label",
#     maxIter=50,
#     maxDepth=5,
#     stepSize=0.1,
#     subsamplingRate=0.8,
#     seed=SEED
# )
# gbt_model = gbt.fit(train_df)
# gbt_time = time.time() - gbt_start

# gbt_pred = gbt_model.transform(test_df)
# gbt_acc = evaluator_acc.evaluate(gbt_pred)
# gbt_f1 = evaluator_f1.evaluate(gbt_pred)

# print(f"    Accuracy: {gbt_acc:.4f} | F1: {gbt_f1:.4f} | Time: {gbt_time:.1f}s")
# results.append(("GBT Classifier", gbt_acc, gbt_f1, gbt_time))


# print("="*60)
# print("MODEL COMPARISON")
# print("="*60)
# results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1 Score", "Time (s)"])
# print(results_df.to_string(index=False))

# best_model_name = results_df.loc[results_df["F1 Score"].idxmax(), "Model"]
# print(f"✓ Best Model: {best_model_name}")

# # Use Random Forest for feature importance by default
# best_model = rf_model

# # ========================================
# # Feature Importance Analysis
# # ========================================
# print("Analyzing feature importance...")

# # Get feature importance from Random Forest
# feature_importance = best_model.featureImportances.toArray()

# # Build proper feature name mapping
# # The feature vector is: [numeric_features] + [one-hot encoded categorical features]
# feature_names = []

# # Add numeric features
# numeric_features_used = [col for col in numeric_features if col in df_classification.columns]
# feature_names.extend(numeric_features_used)

# # Add encoded categorical features (one-hot creates multiple columns per feature)
# # We need to get the actual vector size for each encoded feature
# for col in categorical_features:
#     if col in df_classification.columns:
#         # Get unique values count to determine one-hot vector size
#         num_categories = df_classification.select(col).distinct().count()
#         # One-hot encoding creates n-1 columns for n categories (drop last)
#         for i in range(num_categories - 1):
#             feature_names.append(f"{col}_cat_{i}")

# # Truncate or pad feature names to match importance array length
# if len(feature_names) > len(feature_importance):
#     feature_names = feature_names[:len(feature_importance)]
# elif len(feature_names) < len(feature_importance):
#     # Add generic names for remaining features
#     for i in range(len(feature_names), len(feature_importance)):
#         feature_names.append(f"feature_{i}")

# print(f"Total features in model: {len(feature_importance)}")
# print(f"Feature name mapping: {len(feature_names)} names")

# importance_df = pd.DataFrame({
#     "feature": feature_names,
#     "importance": feature_importance
# })
# importance_df = importance_df.sort_values("importance", ascending=False)

# print("Top 15 Most Important Features:")
# print(importance_df.head(15).to_string(index=False))

# # ========================================
# # Model Training Complete
# # ========================================
# print("="*60)
# print("MODEL TRAINING COMPLETE")
# print("="*60)
# print(f"Trained models: {len(results)}")
# print("Next: Run VISUALIZATION cell to generate plots")

In [ ]:
# # CLASSIFICATION VISUALIZATION: training_inference
# # ========================================
# # Generate visualizations from trained classification models

# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import plotly.express as px
# from sklearn.decomposition import PCA
# from sklearn.metrics import confusion_matrix

# print("="*60)
# print("CLASSIFICATION VISUALIZATION")
# print("="*60)

# # [6/8] Sample for Dimensionality Reduction
# # ========================================
# print(f"\n[6/8] Sampling {SAMPLE_FOR_VIZ*100:.0f}% for dimensionality reduction...")

# SAMPLE_FOR_VIZ_ADJUSTED = min(SAMPLE_FOR_VIZ, 0.0001)  # Max 0.01% of data

# # Sample for visualization
# df_sample = df_ml.sample(False, SAMPLE_FOR_VIZ_ADJUSTED, seed=SEED)
# sample_pd = df_sample.select("features", "label", TARGET_COL).toPandas()

# print(f"Sample size: {len(sample_pd):,} rows")

# # Convert Spark vectors to numpy arrays
# from pyspark.ml.linalg import Vectors
# X_sample = np.array([np.array(row.features.toArray()) for row in sample_pd.itertuples()])
# y_sample = sample_pd[TARGET_COL].values

# print(f"✓ Converted to numpy: {X_sample.shape}")

# # ========================================
# # [7/8] Dimensionality Reduction
# # ========================================
# print("\n[7/8] Performing dimensionality reduction...")

# # PCA
# print("  Running PCA...")
# pca = PCA(n_components=2, random_state=SEED)
# X_pca = pca.fit_transform(X_sample)
# print(f"    Explained variance: {pca.explained_variance_ratio_.sum():.2%}")

# # UMAP (requires umap-learn package)
# print("  Running UMAP...")
# try:
#     import umap
#     umap_model = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1)
#     X_umap = umap_model.fit_transform(X_sample)
#     print(f"    ✓ UMAP complete")
#     has_umap = True
# except ImportError:
#     print("    ⚠️  umap-learn not installed. Installing...")
#     try:
#         install_and_import('umap-learn', 'umap')
#         import umap
#         umap_model = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1)
#         X_umap = umap_model.fit_transform(X_sample)
#         print(f"    ✓ UMAP complete")
#         has_umap = True
#     except:
#         print("    ⚠️  Could not install umap-learn. Skipping UMAP.")
#         X_umap = None
#         has_umap = False

# # ========================================
# # [8/8] Visualization and Saving
# # ========================================
# print("\n[8/8] Generating visualizations...")

# # Create output directory
# import os
# os.makedirs("classification_analysis", exist_ok=True)

# # 1. Feature Importance Plot
# fig, ax = plt.subplots(figsize=(10, 8))
# top_15 = importance_df.head(15)
# ax.barh(range(len(top_15)), top_15["importance"].values)
# ax.set_yticks(range(len(top_15)))
# ax.set_yticklabels(top_15["feature"].values)
# ax.set_xlabel("Importance")
# ax.set_title(f"Top 15 Feature Importance ({best_model_name})")
# ax.invert_yaxis()
# plt.tight_layout()
# plt.savefig("classification_analysis/feature_importance.png", dpi=150, bbox_inches="tight")
# print("  ✓ Saved: classification_analysis/feature_importance.png")
# plt.close()

# # 2. PCA Scatter Plot
# fig = px.scatter(
#     x=X_pca[:, 0], y=X_pca[:, 1],
#     color=y_sample,
#     labels={"x": f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", 
#             "y": f"PC2 ({pca.explained_variance_ratio_[1]:.1%})"},
#     title="PCA: training_inference Classification",
#     color_discrete_sequence=px.colors.qualitative.Set1
# )
# fig.write_html("classification_analysis/pca_scatter.html")
# print("  ✓ Saved: classification_analysis/pca_scatter.html")

# # 3. UMAP Scatter Plot (if available)
# if has_umap:
#     fig = px.scatter(
#         x=X_umap[:, 0], y=X_umap[:, 1],
#         color=y_sample,
#         labels={"x": "UMAP 1", "y": "UMAP 2"},
#         title="UMAP: training_inference Classification",
#         color_discrete_sequence=px.colors.qualitative.Set1
#     )
#     fig.write_html("classification_analysis/umap_scatter.html")
#     print("  ✓ Saved: classification_analysis/umap_scatter.html")

# # 4. Confusion Matrix (using best model predictions on test set)
# print("\nGenerating confusion matrix...")
# pred_pd = rf_pred.select("label", "prediction", TARGET_COL).toPandas()

# # Get label mapping
# label_to_name = pred_pd.groupby("label")[TARGET_COL].first().to_dict()
# sorted_labels = sorted(label_to_name.keys())
# label_names = [label_to_name[i] for i in sorted_labels]

# cm = confusion_matrix(pred_pd["label"], pred_pd["prediction"], labels=sorted_labels)

# fig, ax = plt.subplots(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
#             xticklabels=label_names, yticklabels=label_names, ax=ax)
# ax.set_xlabel("Predicted")
# ax.set_ylabel("Actual")
# ax.set_title(f"Confusion Matrix ({best_model_name})")
# plt.tight_layout()
# plt.savefig("classification_analysis/confusion_matrix.png", dpi=150, bbox_inches="tight")
# print("  ✓ Saved: classification_analysis/confusion_matrix.png")
# plt.close()

# # 5. Model Comparison Bar Chart
# fig, ax = plt.subplots(figsize=(10, 6))
# x = np.arange(len(results_df))
# width = 0.35
# ax.bar(x - width/2, results_df["Accuracy"], width, label="Accuracy")
# ax.bar(x + width/2, results_df["F1 Score"], width, label="F1 Score")
# ax.set_ylabel("Score")
# ax.set_title("Model Comparison: training_inference Classification")
# ax.set_xticks(x)
# ax.set_xticklabels(results_df["Model"], rotation=15, ha="right")
# ax.legend()
# ax.grid(axis="y", alpha=0.3)
# plt.tight_layout()
# plt.savefig("classification_analysis/model_comparison.png", dpi=150, bbox_inches="tight")
# print("  ✓ Saved: classification_analysis/model_comparison.png")
# plt.close()

# # 6. Classification Report
# print("\nClassification Report:")
# print(classification_report(pred_pd["label"], pred_pd["prediction"], 
#                           target_names=label_names, digits=4))

# print("\n" + "="*60)
# print("CLASSIFICATION ANALYSIS COMPLETE")
# print("="*60)
# print(f"\nResults saved in: classification_analysis/")
# print("  - feature_importance.png")
# print("  - pca_scatter.html (interactive)")
# if has_umap:
#     print("  - umap_scatter.html (interactive)")
# print("  - confusion_matrix.png")
# print("  - model_comparison.png")

# print(f"\nBest Model: {best_model_name}")
# print(f"Test Accuracy: {results_df.loc[results_df['Model'] == best_model_name, 'Accuracy'].values[0]:.4f}")
# print(f"Test F1 Score: {results_df.loc[results_df['Model'] == best_model_name, 'F1 Score'].values[0]:.4f}")

# # Clean up
# train_df.unpersist()
# test_df.unpersist()
# df_ml.unpersist()

# print("\n✓ Memory released")





In [ ]:
# ========================================
# FINAL VALIDATION: Read-back from Nessie tables
# ========================================
print("=" * 80)
print("FINAL VALIDATION: Reading back from written Nessie tables")
print("=" * 80)

# Check raw imputed table
print("[1/3] Checking raw imputed table...")
df_check_raw = ness.sql("select * from sandbox.sandbox_finance.dcgm_metrics_raw_imputed_v2 limit 100")

# Verify columns exist
raw_cols = df_check_raw.columns
print(f"Total columns in raw table: {len(raw_cols)}")

if "region_rollup" in raw_cols:
    print("✓ region_rollup persisted in raw table")
else:
    print("✗ ERROR: region_rollup NOT in raw table")
    
if "gen" in raw_cols:
    print("✓ gen persisted in raw table")
else:
    print("✗ ERROR: gen NOT in raw table")

print("Sample from raw table with new columns:")
df_check_raw.select("region", "region_rollup", "product_resolved", "gen", "is_training").show(10, truncate=False)

# Check summary daily table
print("[2/3] Checking summary daily table...")
df_check_summary = ness.sql("select * from sandbox.sandbox_finance.dcgm_metrics_summary_imputed_daily_v2 limit 100")

# Verify columns exist
summary_cols = df_check_summary.columns
print(f"Total columns in daily summary table: {len(summary_cols)}")

if "region_rollup" in summary_cols:
    print("✓ region_rollup persisted in daily summary table")
else:
    print("✗ ERROR: region_rollup NOT in daily summary table")
    
if "gen" in summary_cols:
    print("✓ gen persisted in daily summary table")
else:
    print("✗ ERROR: gen NOT in daily summary table")

# Print schema of daily summary table
print("" + "=" * 50)
print("DAILY SUMMARY TABLE SCHEMA:")
print("=" * 50)
df_check_summary.printSchema()

print("Sample from daily summary table with new columns:")
df_check_summary.select(
    "day", "region", "region_rollup", "gen", 
    "product_resolved", "is_training", "gpu_util_p50"
).show(10, truncate=False)

# Check summary monthly table
print("[3/3] Checking summary monthly table...")
df_check_summary_monthly = ness.sql("select * from sandbox.sandbox_finance.dcgm_metrics_summary_imputed_monthly_v2 limit 100")

# Verify columns exist
summary_monthly_cols = df_check_summary_monthly.columns
print(f"Total columns in monthly summary table: {len(summary_monthly_cols)}")

if "region_rollup" in summary_monthly_cols:
    print("✓ region_rollup persisted in monthly summary table")
else:
    print("✗ ERROR: region_rollup NOT in monthly summary table")
    
if "gen" in summary_monthly_cols:
    print("✓ gen persisted in monthly summary table")
else:
    print("✗ ERROR: gen NOT in monthly summary table")

# Print schema of monthly summary table
print("" + "=" * 50)
print("MONTHLY SUMMARY TABLE SCHEMA:")
print("=" * 50)
df_check_summary_monthly.printSchema()

print("Sample from monthly summary table with new columns:")
df_check_summary_monthly.select(
    "day", "region", "region_rollup", "gen", 
    "product_resolved", "is_training", "gpu_util_p50"
).show(10, truncate=False)


print("" + "=" * 80)
print("✓ FINAL VALIDATION COMPLETE")
print("=" * 80)
print("Downstream notebooks can now use these segment dimensions:")
print("  - gen")
print("  - region_rollup")
print("  - customer_segment")
print("  - product_segment")
print("  - is_training")
print("=" * 80)


In [ ]:
spark.catalog.clearCache()

try:
    if "spark" in globals() and spark is not None:
        spark.stop()
        print("Spark session stopped.")
    elif "ness" in globals() and hasattr(ness, "spark") and ness.spark is not None:
        ness.spark.stop()
        print("Nessie/Spark session stopped.")
    else:
        print("No active Spark session found.")
except Exception as e:
    print(f"Error stopping Spark session: {e}")


# 📋 Notebook Update Summary

## Changes Made for Downstream Segmentation Support

This notebook has been updated to support deeper segmentation analysis in downstream EDA notebooks.

### 1. New Feature Engineering

**Added in Cell 6 (Feature Engineering):**

####  
- **Logic**: 
  - If  starts with "EU" → 
  - Otherwise → 
- **Purpose**: Provides a normalized regional dimension for segmentation
- **Location**: Feature engineering pipeline (after  feature)

####  (already existed, now confirmed in summary)
- **Logic**:
  - If  in ("GB200", "GB300", "B200", "B300") → 
  - Otherwise → 
- **Purpose**: Segments GPUs by generation for analysis

---

### 2. Summary Table Updates

**Modified in Cell 16 (Summary SQL):**

The  query has been updated to include  and  throughout:

1. **Base CTE**: Added  and  to SELECT
2. **per_ts CTE**: Added to SELECT and GROUP BY
3. **tflops_daily CTE**: Added to SELECT and GROUP BY (both inner and outer)
4. **node_daily CTE**: Added to SELECT and GROUP BY
5. **final_rowstats CTE**: Added to SELECT and GROUP BY
6. **final CTE JOIN conditions**: Added to JOIN predicates for  and 

---

### 3. Validation

**Added validation cells:**

1. **Cell 15**: Validates  and  in 
   - Schema checks
   - Data sanity checks (EU/NAM mapping)
   - Distribution analysis

2. **Cell 18** (inline): Validates  and  in 
   - Schema checks
   - Sample data inspection
   - Distribution analysis

3. **Cell 23**: Read-back validation from Nessie tables
   - Confirms persistence in raw imputed table
   - Confirms persistence in summary table
   - Shows sample data with new dimensions

---

### 4. What Downstream Notebooks Can Now Use

The following **segment dimensions** are now available in both the cleaned raw table and summary table:

✅  - GPU generation (current_gen | prior_gen)  
✅  - Regional grouping (EU | NAM)  
✅  - Customer segment (already existed)  
✅  - Product segment (already existed)  
✅  - Training/inference flag (already existed)

---

### 5. Downstream Implementation Notes

**For the Spark EDA notebook ():**

- Use these dimensions for one-at-a-time segmentation
- Implement  and  pattern
- Create segmented versions of:
  - Correlation matrices (p50 and p95)
  - Density plots
  - Monthly drift visualizations (trailing 15 months only)
  - Drift score heatmaps (using IQR-scaled drift)

**For the visualization notebook ():**

- Exclude high-scale metrics from box plots: , , , , 
- Show only trailing 15 months for drift visualizations
- Support correlation visualizations for both p50 and p95

---

### 6. Table Output Locations

- **Raw imputed table**: 
- **Summary table**: 

Both tables now include  and  as normalized segmentation dimensions.

---

**✅ Update Complete**: This notebook now provides the foundational feature engineering and normalized dimensions required for the enhanced downstream segmentation analysis.
